# Step 4 記述統計と文体計量 — MFW・Delta・PCA・特徴語

> **用語の予習・復習**: `docs/glossary.md` の Step 4 を参照。キーワードを見て自分で説明してみてから読むこと。

## このステップの到達目標

1. 最頻語（MFW）による文体計量の考え方を説明できる
2. Burrows's Delta を自分で実装・解釈できる
3. 対数尤度比による特徴語抽出ができ，その限界を言える
4. **word embedding に進む前に，頻度で何が見えるかを確定させる**

## 導入：なぜ最頻語なのか

文体計量（stylometry）の中心的な発見は逆説的である。

> 作者を最もよく識別するのは，**内容と無関係な機能語**の頻度である。

「の」「は」「て」「が」の使用比率は，作家ごとに驚くほど安定し，
主題が変わっても変わらない。だから作者推定に使える。
逆に，**時代やジャンルを見たいときは，機能語が作家効果を持ち込む**。

本コーパスは**同じ作家の作品を複数含む**（森鴎外7点，岡本綺堂6点，
坂口安吾・海野十三・島崎藤村が各5点…）。作家効果が強く出る構造であり，
時代やジャンルの効果を見たいときはこれが交絡する。
なお島崎藤村が9点に見えていたのは『夜明け前』4冊・『家』2冊を
別作品として数えていたからで，Step 2 の分冊結合で解消してある。
このステップで「何が作家由来で，何が時代由来か」を切り分ける感覚を作る。

## 参考
- Burrows, J. (2002) 'Delta': a measure of stylistic difference. *LLC* 17(3).
- Evert, S. et al. (2017) Understanding and explaining Delta measures. *DSH* 32.
- 金明哲 (2021)『テキストアナリティクス』共立出版


In [ ]:
# ---- 共通の準備（毎回このセルから実行する）----------------------------
import os, sys, csv, json, math, random, shutil, subprocess, warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore', category=FutureWarning)

# リポジトリのルートを自動で探す（my_work/notebooks/ でも notebooks/ でも，上へたどる）
ROOT = Path.cwd()
while not (ROOT / 'config' / 'pipeline.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)
if Path.cwd().resolve() == (ROOT / 'notebooks').resolve():
    print('[注意] 配布版（notebooks/）を直接開いている。実行すると次の git pull が止まる。\n'
          '       python scripts/copy_notebooks.py でコピーを作り，my_work/notebooks/ の方を開くこと。')

# 日本語フォント（□ にならないように）
for cand in ['Hiragino Sans', 'Yu Gothic', 'Meiryo',
             'Noto Sans CJK JP', 'IPAexGothic', 'MS Gothic']:
    if cand in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = cand
        break
else:
    print('[!] 日本語フォントが見つかりません。docs/00_setup_students.md §1.7 を参照。')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ---- 図はすべて SVG（ベクタ）で保存する ---------------------------------
# 論文・スライドに載せる図は拡大しても劣化してはならない。PNG は解像度が
# 固定されるので，投影や印刷で文字が潰れる。SVG なら任意の倍率で鮮明で，
# Illustrator / Inkscape で軸ラベルだけを直すこともできる。
FIG_EXT   = 'svg'
RASTER_DPI = 200          # rasterized=True の要素にだけ効く
plt.rcParams['svg.fonttype']       = 'path'   # 文字をアウトライン化して環境非依存に
plt.rcParams['savefig.transparent'] = False
# 画面へのインライン表示は既定（PNG）のままにする。
# InlineBackend.figure_formats を 'svg' に変えると，JupyterLab や
# VS Code の版によっては図がまったく表示されなくなることがある。
# **保存されるファイルは SVG** なので，論文・スライドに使うほうは
# ベクタで手元に残る。画面で拡大して見たいときは save_fig が表示する
# パスの .svg をブラウザで開くこと。

def need(path, hint=''):
    """必要な入力があるか確かめる。無ければ**理由を表示して** False を返す。

    セルを `if p.exists():` で囲むと，入力が無いときに何も起きない。
    学生には「壊れている」と「まだ前の工程を走らせていない」の区別が
    つかず，図が出ないという相談の大半がこれである。必ず理由を出す。
    """
    p = Path(path)
    try:
        ok = p.is_file() or (p.is_dir() and any(p.iterdir()))
    except OSError:
        ok = False
    if not ok:
        print(f'[未実行] {p} がありません。')
        if hint:
            print(f'         {hint}')
        print('         この Step の前のセルを上から順に実行すること。'
              '\n         それでも出ない場合は，前の Step のノートブックが'
              '最後まで通っているか確認する。')
    return ok


# 旧名 → 新名。**中身は 0–1 の割合なので per cent は誤称**であった。
# 2026-09-24 に改名。古い出力を持っている人のために読み替えだけは残す。
LEGACY_COLS = {'df_all_pct': 'df_all_prop', 'df_in_pct': 'df_in_prop'}


def read_table(path, **kw):
    """CSV を読み，**古い列名があれば新しい名前に読み替える**。

    列名の約束：割合（0–1）は ``_prop`` / ``_ratio`` / ``_share``，
    百分率（0–100）だけを ``_pct`` と綴る。``df_all_prop`` が 0.1584 なら
    15.84 % の意である。読み替えたときは黙らずに知らせる — 黙って直すと，
    手元の CSV と教材の列名が食い違っていることに気づけないため。
    """
    d = pd.read_csv(path, **kw)
    old = {k: v for k, v in LEGACY_COLS.items()
           if k in d.columns and v not in d.columns}
    if old:
        d = d.rename(columns=old)
        print('[note] 古い列名を読み替えた: '
              + '，'.join(f'{k}→{v}' for k, v in old.items())
              + '\n       07_descriptive_stats.py を走らせ直すと'
                '新しい名前で書き出される。')
    return d


def load_meta(path=None, analysis_only=True):
    """メタデータを読む。既定では**分析に使う行だけ**を返す。

    落とすのは2種類。書誌としては残すが，集計に足してはいけない行である。
      superseded … v1 の合本。増補で分冊ごとに取り直したので，足すと
                   同じ作品を二重に数える
      merged     … 分冊。03b で canonical の巻に本文を統合したので，
                   この行はもう本文を持たない（『夜明け前』『家』）
      too_short  … 1チャンクにも満たず，チャンク単位の分析に乗らない

    生の表がほしいときは ``analysis_only=False``。
    """
    df = pd.read_csv(path or META)
    if analysis_only and 'completeness' in df.columns:
        drop = df['completeness'].isin(['superseded', 'merged', 'too_short'])
        if drop.any():
            names = '，'.join(df.loc[drop, 'title_aozora'].astype(str))
            print(f'[meta] 分析から除外 {int(drop.sum())} 行: {names}')
        df = df[~drop].reset_index(drop=True)
    return df


def w_ljust(text, width):
    """全角を2桁と数えて左詰めする。

    ``f'{s:<26}'`` は**文字数**で詰めるので，日本語の作品名を並べると
    桁が揃わない（全角は2桁ぶんの幅を占める）。表として読ませるなら
    表示幅で詰めること。

    **なお，一覧を出すなら ``show()`` で表にするほうがよい**（下記）。
    この関数は，表にしにくいもの（KWIC の前後文脈など）を print で
    並べるときに使う。
    """
    import unicodedata
    text = str(text)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    return text + ' ' * max(0, width - w)


# ---------------------------------------------------------------------------
# 分析結果の表示
# ---------------------------------------------------------------------------
# **一覧は print ではなく表で出す。**
#   * print は桁が揃わない（全角の幅）。数字の比較がしにくい
#   * 列に名前が付かないので，あとで見返したときに何の数字か分からない
#   * 並べ替えも絞り込みもできない
# 表にすると，列名がそのまま「何を測ったか」の記録になる。
# **ただし何でも表にするのではない。** 単発の数値・警告・KWIC の前後文脈は
# 文のほうが読みやすい。目安は「2列以上あるか」「行が並ぶか」。
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('caption-side', 'top'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '0 0 .4em 0'),
               ('color', '#33322e'), ('font-size', '95%')]},
    {'selector': 'th',
     'props': [('background', '#f2f2ef'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '.26em .7em'),
               ('border-bottom', '1px solid #c6c5bd'), ('white-space', 'nowrap')]},
    {'selector': 'td',
     'props': [('padding', '.22em .7em'), ('border-bottom', '1px solid #ecebe6')]},
    {'selector': 'tbody tr:hover td', 'props': [('background', '#f7f7f4')]},
]


def show(df, caption='', fmt=None, index=False, header=True, na='—', align=None):
    """DataFrame を表として表示する。Jupyter 以外でも落ちない。

    ``fmt`` は pandas の ``Styler.format`` に渡す辞書
    （例 ``{'一致率': '{:.1%}', 'G²': '{:.0f}'}``）。
    数値の列は自動で右寄せにする。``align`` で列ごとに寄せを指定できる。
    KWIC の左文脈を ``align={'左文脈': 'right'}`` にすると，
    **キーワードが縦に揃う**（等幅フォントに頼らずに揃う）。
    返り値は ``df`` なので ``t = show(df)`` として続けて使える。
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    try:
        from IPython.display import display as _display
        st = df.style.format(fmt, na_rep=na) if fmt else df.style.format(na_rep=na)
        st = st.set_table_styles(TABLE_STYLES)
        num = list(df.select_dtypes('number').columns)
        if num:
            st = st.set_properties(subset=num, **{'text-align': 'right'})
        for col, side in (align or {}).items():
            if col in df.columns:
                st = st.set_properties(subset=[col],
                                       **{'text-align': side,
                                          'white-space': 'pre'})
        if caption:
            st = st.set_caption(caption)
        if not index:
            st = st.hide(axis='index')
        if not header:
            st = st.hide(axis='columns')
        _display(st)
    except Exception:                                   # noqa: BLE001
        # ノートブックの外（スクリプトから import したとき）でも読める形
        if caption:
            print(caption)
        print(df.to_string(index=index, header=header))
    return df


def grid(items, ncol=8, caption=''):
    """語の並びを ``ncol`` 列の表にして表示する。

    40 語を1行に流すと折り返しで読めない。列に切ると目で追える。
    順位が要るなら ``show()`` に順位列を付けた表を渡すこと。
    """
    items = [str(x) for x in items]
    rows = [items[i:i + ncol] for i in range(0, len(items), ncol)]
    rows = [r + [''] * (ncol - len(r)) for r in rows]
    t = pd.DataFrame(rows, columns=[f'_{i}' for i in range(ncol)])
    return show(t, caption=caption, header=False)


def work_rows(meta_df=None):
    """``work_stem`` からメタデータの行を引く辞書を作る。

    ``meta_df`` を省くと**分析対象外の行も含めた全件**から作る。
    表示用の名前は，分析から外した作品についても引けるほうがよい。

    **キーの綴りに注意。** 青空文庫の作品 ID は索引では 0 埋めされていない
    （``1743``）が，本パイプラインのファイル名は6桁に 0 埋めしてある
    （``000119_001743``）。素朴に連結すると ``000119_1743`` となり，
    **1件も一致しない**。辞書は空振りしても例外を出さないので，
    誰の何だか分からないまま最後まで通ってしまう。両方の綴りを登録する。

    ``file_v1`` は増補 45 点では空である。``os.path.splitext(nan)`` は
    例外になるので，文字列であることを確かめてから使う。
    """
    if meta_df is None:
        meta_df = load_meta(analysis_only=False)
    d = {}
    for _, r in meta_df.iterrows():
        fv = r.get('file_v1')
        if isinstance(fv, str) and fv.strip():
            d[os.path.splitext(fv)[0]] = r
        pid = str(r.get('aozora_person_id') or '').strip()
        wid = str(r.get('aozora_work_id') or '').strip()
        if pid and wid and pid.lower() != 'nan' and wid.lower() != 'nan':
            for k in (f'{pid.zfill(6)}_{wid.zfill(6)}',
                      f'{pid}_{wid}', f'{pid.zfill(6)}_{wid}'):
                d[k] = r
    return d


def work_labels(meta_df=None, maxlen=12, with_year=False):
    """``work_stem`` → ``作者『作品』`` の対応表を返す。

    ``000119_001743`` と出されても誰の何だか分からない。距離の近い
    ペアを見るときに**どの作家のどの作品か**が分からなければ，
    「作家効果か時代効果か」という問いにそもそも答えられない。
    表示するときは必ずこれを通すこと。
    """
    out = {}
    for k, r in work_rows(meta_df).items():
        t = str(r.get('title_aozora') or '')
        lab = f"{r.get('author_ja', '?')}『{t[:maxlen]}』"
        if with_year and str(r.get('year_first') or '').strip():
            lab += f"({r['year_first']})"
        out[k] = lab
    return out


def attach_meta(df, cols, stem_col='work_stem', meta_df=None, quiet=False,
                fill_blank=True):
    """``df`` に足りないメタデータの列を，``work_stem`` から引いて補う。

    ``fill_blank=True``（既定）なら，**列はあるのに値が空**のセルも補う。
    列が無いより，列があって半分が空のほうが危ない。列が無ければ
    ``AttributeError`` で止まるが，値が空だと**図がそのまま描けてしまう**。
    2026-09-22 に 07 の突合が外れ，101 点のうち 62 点の ``year_first`` が
    空になった。図は描けたが，62 点が「初出年不明」の灰色で並んだ。
    値の空きも数えて報告し，ここで補えるものは補う。

    分析スクリプトの出力は，その分析に要る列しか書かない。
    ``09_doc2vec.py`` の ``work_vectors.csv`` に ``genre_main`` が無いのは
    その一例である。ノートブックで ``wv.genre_main`` と書けば
    ``AttributeError: 'DataFrame' object has no attribute 'genre_main'``
    になるが，**足りないのは列であって情報ではない**。
    メタデータ表には必ずあるのだから，ここで引いて補えばよい。

    出力 CSV の列構成に図の描画が依存するのは弱い。分析スクリプトを
    書き換えるたびに図が落ちる。図の側で「要る列を宣言して取りに行く」
    ほうが，どちらを先に走らせても通る。

    引けなかった列は空のまま残し ``[warn]`` を出す。図が落ちるより，
    「この軸は色分けできなかった」と分かったうえで出るほうがよい。
    """
    df = df.copy()
    if stem_col not in df.columns:
        if not quiet:
            print(f'[warn] {stem_col} 列が無いので補完できない: {list(cols)}')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        return df

    rows = work_rows(meta_df)

    # **まずキーが合っているかを見る。** 合っていなければ何も補えない。
    # 「1件も合わない」のはたいてい 0 埋めの綴り違いで，黙って通すと
    # 全部の軸が空のまま図になる。
    stems = df[stem_col].astype(str)
    found = stems.map(lambda s: s in rows)
    if not quiet and not found.all():
        n_miss = int((~found).sum())
        lv = 'FATAL' if found.sum() == 0 else 'warn '
        print(f'[{lv}] {stem_col} がメタデータと突合できない行が '
              f'{n_miss}/{len(df)} 件ある: '
              + '，'.join(stems[~found].head(4)))
        if found.sum() == 0:
            print('        **1件も合っていない。** 作品 ID の 0 埋めの'
                  '綴り違いを疑うこと（例 000119_1743 と 000119_001743）。')
            print('        この表を作ったスクリプトのキーの作り方を直すこと。')

    def _blank(v):
        return v is None or str(v).strip().lower() in ('', 'nan', 'none')

    for c in cols:
        if c not in df.columns:
            vals = [(lambda r: '' if r is None or _blank(r.get(c))
                     else r.get(c))(rows.get(s)) for s in stems]
            df[c] = vals
            n = int(sum(1 for v in vals if str(v).strip()))
            if not quiet:
                mark = 'ok  ' if n == len(df) else 'warn'
                print(f'[{mark}] {c} をメタデータから補完: {n}/{len(df)} 件')
            continue

        if not fill_blank:
            continue
        # 列はある。空のセルだけを埋める。
        blank = df[c].map(_blank)
        if not blank.any():
            continue
        filled = 0
        vals = df[c].tolist()
        for i, (s, is_blank) in enumerate(zip(stems, blank)):
            if not is_blank:
                continue
            r = rows.get(s)
            if r is not None and not _blank(r.get(c)):
                vals[i] = r.get(c)
                filled += 1
        df[c] = vals
        if not quiet:
            left = int(sum(1 for v in vals if _blank(v)))
            mark = 'fix ' if left == 0 else 'warn'
            print(f'[{mark}] {c} は {int(blank.sum())}/{len(df)} 件が空だった'
                  f' → {filled} 件をメタデータから補完'
                  + ('' if left == 0 else f'（なお {left} 件が空）'))
            if left:
                print('        **その列で色分けする図・集計は，この件数を'
                      '報告に書くこと。**')
    return df


def label_points(ax, xs, ys, texts, fontsize=8, color='#333333', pad=4,
                 leader='line', leader_min=13, crowd_r=26,
                 leader_color='#8a8a83'):
    """散布図の注記を，重ならない位置だけに置き，遠いものは引き出し線で結ぶ。

    素朴に ``ax.annotate(t, (x, y))`` と書くと，**注目すべき点ほど一箇所に
    固まる**ので注記が必ず重なって読めなくなる。文語標識の上位は
    どれも口語標識がほぼ 0 で，対数軸の右下隅に密集するのが典型である。

    そこで点の周囲を順に試し，他の注記とも他の点とも重ならず，かつ軸の
    内側に収まる位置があればそこに置く。どこにも置けない注記は**置かずに
    数だけ報告する**。読めない字を重ねるより，図の外（下の表）で番号から
    引くほうがよい。

    **離れた位置に置いた注記は，引き出し線で点と結ぶ。** 避けた結果として
    注記は点から離れるので，線が無いとどの点の名前なのか分からなくなる
    ——密集した領域では隣の点の名前だと読まれる。線があれば，遠くへ逃がす
    ことに副作用が無くなるので，**近くに空きが無い注記も置ける**ようになる
    （候補の輪を 24・30 ポイントまで広げてあるのはそのため）。

    ``leader``
        ``'line'``（既定）… 矢じりの無い細線で結ぶ。図版の慣例はこちら。
        6.5pt の文字に矢じりを付けると，マーカーそのものを覆って点が読めなくなる
        ``'arrow'`` … 小さな矢じりを付ける
        ``'none'`` … 結ばない（従来どおり）
    ``leader_min``
        この距離（ポイント）より遠くに置いた注記を結ぶ。既定は 0，
        つまり**すべて結ぶ**。注記は必ず点から離れた位置に置かれるので，
        離れている以上「どの点の名前か」は線でしか確定しない。
        線を省くと，隣の点の名前だと読まれる余地が残る。
    ``crowd_r``
        注記の近くに**自分以外の点**がこの半径（ピクセル）内にあるかを
        見る。``leader_min`` を上げて線を減らしたときでも，
        紛れる相手が居る注記だけは必ず結ぶための保険である。

    表示座標で矩形の重なりを見るので，**軸の位置が確定してから**呼ぶ。
    ``fig.tight_layout()`` はこの関数より**前**に呼ぶこと（後で呼ぶと軸が
    動き，せっかく避けた位置がずれる）。戻り値は置けた注記の数。
    """
    from matplotlib.transforms import Bbox
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    texts = list(texts)
    if len(texts) == 0:
        return 0
    # **長さが違えば黙って切り詰めずに止める。** zip は短いほうに合わせるので，
    # 座標だけを絞り込んで名前を絞り忘れると，先頭から順に**別の作品の名前**が
    # 貼られた図が，何の警告も出さずに出来上がる。これがいちばん重い事故である。
    if not (len(xs) == len(ys) == len(texts)):
        raise ValueError(
            f'label_points: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'名前={len(texts)}）。座標と名前を同じ添字で絞り込むこと。'
            'たとえば P[pick,0] と組むのは names[pick] であって names ではない。')
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(renderer=ren)

    def pad_box(b, w=2.0, h=1.5):
        """注記の矩形に**絶対量の余白**を足す。

        倍率（``expanded(1.08, …)``）では足りない。1桁の数字は幅 8px ほど
        なので 8% は 0.6px にしかならず，隣り合う注記が触れるほど近くても
        「重なっていない」と判定される。**``21`` が「21」と読める**のは
        これが原因である。文字の大小によらず一定の余白を確保する。
        """
        return Bbox.from_extents(b.x0 - w, b.y0 - h, b.x1 + w, b.y1 + h)

    def bb_of(ann):
        # Annotation 自身の get_window_extent を使うこと。
        # Text.get_window_extent(ann, ...) を呼ぶと xy の位置が無視され，
        # xytext のオフセットを絶対座標と見た矩形が返って判定が壊れる。
        return ann.get_window_extent(renderer=ren)

    blocked = []
    for coll in ax.collections:
        try:
            for p in coll.get_offsets():
                px, py = ax.transData.transform(p)
                blocked.append(Bbox.from_bounds(px - pad, py - pad,
                                                2 * pad, 2 * pad))
        except Exception:                                    # noqa: BLE001
            pass

    # **まっすぐ真上・真下を先に試す。** 点の直上に中央揃えで置ければ，
    # それがいちばん素直で，引き出し線も要らない。横へずらすのは，
    # 直上が塞がっていたときの次善である。
    #
    # 横へずらす輪は 12 ポイントから始める。**線が線として見える長さを
    # 確保する**ため。8 ポイントに置くと引き出し線が3ピクセルの点にしか
    # ならず，汚れと区別がつかない。
    # **真上に置けなければ，まず真上へ逃がす。** 横へ逃がすと注記の左右の
    # 順序が点の順序と入れ替わり，引き出し線も交差する。真上に段を重ねる
    # 限り，x は動かないので順序は必ず保たれる。横へずらすのは最後。
    # **横のずらし幅は小さく取る。** 横へ 30 ポイントも動かすと，注記が
    # 隣の点の真上に乗り，引き出し線で結んでも読みにくい。真上に段を
    # 重ねるほうが先で（x が動かないので順序が保たれる），横は 8→18
    # ポイントの範囲に収める。
    CAND = [(0, 9), (0, -11), (0, 20), (0, -22), (0, 31), (0, -33),
            (8, 5), (-8, 5), (8, -12), (-8, -12),
            (11, 0), (-11, 0),
            (13, 9), (-13, 9), (13, -16), (-13, -16),
            (18, 0), (-18, 0), (18, 14), (-18, 14),
            (0, 42), (0, -44)]

    def _ha(dx):
        # dx が 0 なら**中央揃え**。ここを 'left' にすると，真上に置いた
        # つもりの注記が文字幅の半分だけ右にずれ，隣の点の上に乗る。
        return 'center' if dx == 0 else ('left' if dx > 0 else 'right')
    placed, chosen, skipped = [], [], 0
    for x, y, t in zip(xs, ys, texts):
        # **自分が指している点は避けない。** 除かないと，注記は必ず
        # 自分の点の隣に来るので全部「重なる」と判定され，1つも置けない。
        ox, oy = ax.transData.transform((x, y))
        near = [b for b in blocked
                if not (abs((b.x0 + b.x1) / 2 - ox) < 1
                        and abs((b.y0 + b.y1) / 2 - oy) < 1)]
        for dx, dy in CAND:
            ann = ax.annotate(str(t), (x, y), textcoords='offset points',
                              xytext=(dx, dy), fontsize=fontsize, color=color,
                              ha=_ha(dx),
                              va='bottom' if dy >= 0 else 'top', zorder=6)
            bb = pad_box(bb_of(ann))
            inside = (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                      and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1)
            if inside and not any(bb.overlaps(b) for b in placed + near):
                placed.append(bb)
                chosen.append((ann, x, y, str(t), dx, dy))
                break
            ann.remove()
        else:
            skipped += 1

    # ---- 交差をほどく ----------------------------------------------------
    # **引き出し線が交差すると，注記の左右の順序が点の順序と入れ替わる。**
    # 文語標識の上位のように順位そのものが意味を持つ図では，2 と 3 が
    # 入れ替わって並ぶだけで読み違えられる。交差している2件を見つけ，
    # **位置を入れ替えて交差が解ければ入れ替える**（2-opt）。
    def _cross(p, q, r, s):
        def o(a, b, c):
            return ((b[0] - a[0]) * (c[1] - a[1])
                    - (b[1] - a[1]) * (c[0] - a[0]))
        return (((o(r, s, p) > 0) != (o(r, s, q) > 0))
                and ((o(p, q, r) > 0) != (o(p, q, s) > 0)))

    kpt = fig.dpi / 72.0

    def _seg(i):
        ann, x, y, t, dx, dy = chosen[i]
        ox, oy = ax.transData.transform((x, y))
        return (ox, oy), (ox + dx * kpt, oy + dy * kpt)

    def _set_off(i, dx, dy):
        ann, x, y, t, _, _ = chosen[i]
        ann.set_position((dx, dy))
        ann.set_ha(_ha(dx))
        ann.set_va('bottom' if dy >= 0 else 'top')
        chosen[i] = (ann, x, y, t, dx, dy)

    def _fits(i, bb):
        ox, oy = ax.transData.transform((chosen[i][1], chosen[i][2]))
        if not (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1):
            return False
        return not any(bb.overlaps(b) for b in blocked
                       if abs((b.x0 + b.x1) / 2 - ox) > 1
                       or abs((b.y0 + b.y1) / 2 - oy) > 1)

    swaps = 0
    for _ in range(3):
        improved = False
        for i in range(len(chosen)):
            for j in range(i + 1, len(chosen)):
                if not _cross(*_seg(i), *_seg(j)):
                    continue
                di, dj = chosen[i][4:6], chosen[j][4:6]
                _set_off(i, *dj)
                _set_off(j, *di)
                bi = pad_box(bb_of(chosen[i][0]))
                bj = pad_box(bb_of(chosen[j][0]))
                others = [b for k2, b in enumerate(placed) if k2 not in (i, j)]
                good = (not bi.overlaps(bj)
                        and not any(bi.overlaps(b) or bj.overlaps(b)
                                    for b in others)
                        and _fits(i, bi) and _fits(j, bj)
                        and not _cross(*_seg(i), *_seg(j)))
                if good:
                    placed[i], placed[j] = bi, bj
                    swaps += 1
                    improved = True
                    continue
                _set_off(i, *di)
                _set_off(j, *dj)

                # 入れ替えが収まらないときは，**片方を別の候補位置へ動かす**。
                # 入れ替えは2つの箱の大きさが違うと失敗しやすい（数字1桁と
                # 作者名では幅が違う）。動かすほうは箱の大きさが変わらない。
                moved = False
                for who, other in ((i, j), (j, i)):
                    d0 = chosen[who][4:6]
                    for cx, cy in CAND:
                        if (cx, cy) == tuple(d0):
                            continue
                        _set_off(who, cx, cy)
                        bw = pad_box(bb_of(chosen[who][0]))
                        rest = [b for k2, b in enumerate(placed) if k2 != who]
                        if (_fits(who, bw)
                                and not any(bw.overlaps(b) for b in rest)
                                and not _cross(*_seg(who), *_seg(other))
                                and not any(_cross(*_seg(who), *_seg(k2))
                                            for k2 in range(len(chosen))
                                            if k2 != who)):
                            placed[who] = bw
                            swaps += 1
                            moved = improved = True
                            break
                        _set_off(who, *d0)
                    if moved:
                        break
        if not improved:
            break

    # ---- 引き出し線 ------------------------------------------------------
    # **線は配置が全部決まってから付ける。** arrowprops を付けた
    # Annotation の get_window_extent は「文字＋線」の外接矩形を返すので，
    # 配置の判定に使うと自分の点と必ず重なり，1件も置けなくなる。
    n_leader = 0
    if leader in ('line', 'arrow'):
        style = '-' if leader == 'line' else '-|>'
        for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
            ox, oy = ax.transData.transform((x, y))

            def dist_to_box(px, py, b=bb):
                # 文字の矩形から点までの距離。矩形の中なら 0。
                ddx = max(b.x0 - px, 0, px - b.x1)
                ddy = max(b.y0 - py, 0, py - b.y1)
                return (ddx * ddx + ddy * ddy) ** .5

            # **素直に置けたものには線を引かない。** 点の直上（または直下）に
            # 中央揃えで載っていて，しかもその注記にいちばん近い点が自分の
            # 点であれば，どの点の名前かは見れば分かる。線はかえって邪魔
            # である。横へ逃がしたものだけを結ぶ。
            if dx == 0 and abs(dy) <= 12:
                continue            # 点の真上・真下の一段目 → 線は要らない
            # それ以外は結ぶ。**段を上げたものも結ぶ。** 一段上げた注記の
            # 真下には別の点の注記が入るので，どちらの点のものか分からなく
            # なる。横へずらしたものは言うまでもない。
            d_other = min(
                (dist_to_box((b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2)
                 for b in blocked
                 if abs((b.x0 + b.x1) / 2 - ox) > 1
                 or abs((b.y0 + b.y1) / 2 - oy) > 1),
                default=float('inf'))
            if (dx * dx + dy * dy) ** .5 < leader_min and d_other >= crowd_r:
                continue
            ann.remove()
            ax.annotate(t, (x, y), textcoords='offset points',
                        xytext=(dx, dy), fontsize=fontsize, color=color,
                        ha=_ha(dx),
                        va='bottom' if dy >= 0 else 'top', zorder=6,
                        arrowprops=dict(arrowstyle=style, linewidth=.55,
                                        color=leader_color, alpha=.9,
                                        shrinkA=1.5, shrinkB=2.5,
                                        mutation_scale=7))
            n_leader += 1

    # ---- 誤読の自己点検 --------------------------------------------------
    # **注記の最寄りの点が自分の点でないものを数える。** これが
    # 「ラベルとデータ点がずれて見える」の正体である。引き出し線を
    # 引いてあれば誤読にはならないが，線を切った設定では危険なので，
    # そのときだけ警告を出す。
    risky = []
    for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
        ox, oy = ax.transData.transform((x, y))
        cx, cy = (bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2
        d_own = ((cx - ox) ** 2 + (cy - oy) ** 2) ** .5
        d_other = min(
            (((b.x0 + b.x1) / 2 - cx) ** 2 + ((b.y0 + b.y1) / 2 - cy) ** 2) ** .5
            for b in blocked
            if abs((b.x0 + b.x1) / 2 - ox) > 1 or abs((b.y0 + b.y1) / 2 - oy) > 1
        ) if len(blocked) > 1 else float('inf')
        if d_other < d_own:
            risky.append(t)
    left = sum(1 for i in range(len(chosen)) for j in range(i + 1, len(chosen))
               if _cross(*_seg(i), *_seg(j)))
    if skipped:
        print(f'[fig] 重なるため {skipped} 件の注記を省いた（表で引くこと）')
    if left:
        print(f'[warn] 引き出し線の交差が {left} 件ほどけなかった。'
              '注記の左右の順序が点の順序と食い違う。'
              '注記を短くするか，件数を減らすこと。')
    if risky:
        head = '，'.join(str(r) for r in risky[:6])
        more = f' ほか{len(risky) - 6}件' if len(risky) > 6 else ''
        if leader in ('line', 'arrow'):
            print(f'[fig] {len(risky)} 件の注記は別の点のほうが近い'
                  f'（{head}{more}）。引き出し線で結んであるので読み違えない。')
        else:
            print(f'[warn] {len(risky)} 件の注記は**別の点のほうが近い**'
                  f'（{head}{more}）。leader="none" では読み違えが起きる。')
    return len(placed)
def _proj_versions():
    """射影に関わる版を並べる（うまくいかないときの手がかり）。"""
    import importlib
    out = []
    for nm in ['numpy', 'numba', 'llvmlite', 'pynndescent', 'sklearn']:
        try:
            out.append(f'{nm} ' + str(getattr(importlib.import_module(nm),
                                              '__version__', '?')))
        except Exception:                                    # noqa: BLE001
            out.append(f'{nm} ×')
    return '／'.join(out)

def umap_diagnosis(e):
    """UMAP が使えないときに，**何をすればよいか**を出す。"""
    import sys
    print(f'[NG  ] UMAP が使えない: {type(e).__name__}: {e}')
    print(f'       このカーネルの Python = {sys.executable}')
    print(f'       {_proj_versions()}')
    if isinstance(e, ModuleNotFoundError):
        # **入れた先とカーネルの環境が違う**のが圧倒的に多い。
        # uv add は「プロジェクト（pyproject.toml のある場所）」単位なので，
        # dh_project/pyproject.toml が無い，または dh_project の外に clone
        # した場合は，uv は別のプロジェクトに入れる。カーネルの .venv には入らない。
        print('       **この環境には入っていない。** 入れた先が違う可能性が高い')
        print('       （uv add はプロジェクト単位。~/Documents/dh_project に')
        print('        pyproject.toml が無いと，別のプロジェクトに入る）。')
        print('       この環境を名指しして入れるのが確実:')
        import platform as _pf
        if sys.platform == 'darwin' and _pf.machine() == 'x86_64':
            # **Intel Mac は版を固定する。** llvmlite の x86_64 wheel は
            # 0.45.1 が最後で，0.46 以降は arm64 のみ。固定しないと
            # ソースからのビルドに落ち，Homebrew の LLVM と版が合わずに
            # 失敗する（llvmlite 0.49 は LLVM 22 を要求）。
            print('       （Intel Mac なので**版を固定する**。'
                  'llvmlite の x86_64 wheel は 0.45.1 が最後）')
            print(f'         uv pip install --python "{sys.executable}" \\')
            print('             --only-binary :all: \\')
            print('             "numba==0.62.1" "llvmlite==0.45.1" '
                  '"numpy<2.4" umap-learn')
        else:
            print(f'         uv pip install --python "{sys.executable}" umap-learn')
        print('       入れたら**カーネルを再起動**して，このセルから実行し直す。')
    else:
        print('       import は通るが使えない型の失敗である'
              '（別パッケージの umap／numba と numpy の版違い／'
              'numba のキャッシュ）。')
    print('       切り分けの全項目:')
    print('         import sys, subprocess; print(subprocess.run('
          '[sys.executable,')
    print("             str(ROOT/'scripts'/'check_umap.py')], "
          'capture_output=True,')
    print('             text=True).stdout)')

def project(Xn, how='umap', seed=20260920, n_neighbors=15, min_dist=0.12,
            perplexity=30):
    """高次元の行列を2次元に落とす。**どの方法で落としたかを必ず返す。**

    ``how`` は ``'umap'``／``'tsne'``／``'auto'``。既定の ``'umap'`` は，
    使えなければ**止まって理由を出す**。``'auto'`` のときだけ t-SNE に落ちる。
    **黙って別の方法に替えないのが肝心である**（図は出るが塊の見え方は
    変わるので，環境の問題を分析結果と読み違える）。

    Step 5 の §3（主成分分析との比較）と §4（ギャラクシー）が共有する。
    """
    import importlib
    Xn = np.asarray(Xn, dtype=np.float32)
    if how in ('auto', 'umap'):
        try:
            m = importlib.import_module('umap')
            if not hasattr(m, 'UMAP'):
                # PyPI には umap（別物）と umap-learn（本物）がある。
                # pip install umap をしていると import umap はそちらを拾う。
                raise ImportError(
                    f'umap に UMAP クラスが無い（{getattr(m, "__file__", "?")}）。'
                    '別パッケージの umap が入っている。'
                    'umap を外して umap-learn を入れること')
            P = m.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                       metric='cosine', random_state=seed).fit_transform(Xn)
            return (np.asarray(P, dtype=np.float32),
                    f'UMAP {getattr(m, "__version__", "")}'
                    f' (n_neighbors={n_neighbors}, min_dist={min_dist}, cosine)')
        except Exception as e:                               # noqa: BLE001
            umap_diagnosis(e)
            if how == 'umap':
                # **黙って別の方法に替えない。** どうしても t-SNE で
                # 進めたいときは 'tsne' と明示し，報告にもそう書くこと。
                raise
            print('[warn] how="auto" なので t-SNE に切り替える。'
                  '**図と報告に t-SNE と書くこと。**')
    from sklearn.manifold import TSNE
    P = TSNE(n_components=2, perplexity=perplexity, metric='cosine',
             init='pca', random_state=seed).fit_transform(Xn)
    return (np.asarray(P, dtype=np.float32),
            f't-SNE (perplexity={perplexity}, cosine)')


def proj_quality(Xn, P, k=10):
    """射影がどれだけ嘘をついているかを3つの数で返す。

    ``trust``  … 2次元で近く見える点が原空間でも近いか（局所・1が最良）
    ``keep``   … 原空間の上位 k 近傍のうち画面でも上位 k に入る語数
    ``rho``    … 原空間の距離と画面の距離の順位相関（**大域**の保存）

    局所（trust・keep）と大域（rho）は別物である。**UMAP は局所に強く，
    主成分分析は大域に強い**——これを目で見ずに数で確かめるための関数。
    """
    from scipy.spatial.distance import pdist
    from scipy.stats import spearmanr
    from sklearn.manifold import trustworthiness
    Xn, P = np.asarray(Xn, np.float32), np.asarray(P, np.float32)
    S = Xn @ Xn.T
    np.fill_diagonal(S, -np.inf)
    nn_t = np.argsort(-S, axis=1)[:, :k]
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    nn_p = np.argsort(d2, axis=1)[:, :k]
    keep = np.array([len(set(a) & set(b)) for a, b in zip(nn_t, nn_p)])
    trust = float(trustworthiness(Xn, P, n_neighbors=k, metric='cosine'))
    rho = float(spearmanr(pdist(Xn, 'cosine'), pdist(P))[0])
    return {'trust': trust, 'keep': keep, 'rho': rho}


def reserve_right(fig, frac=0.80):
    """面の外に凡例を置いた図で，**右に余白を確保する**。

    ``tight_layout()`` は面の外に置いた凡例を数えないので，そのままだと
    凡例が図の枠から出る。静止版は ``bbox_inches='tight'`` で救われるが，
    **HTML に埋め込む版は切り取らない**（切り取ると点の位置の割合が
    ずれる）ので，凡例が切れて読めなくなる。実際に切れた。

    ``tight_layout()`` の**後**，注記（``label_points``）の**前**に呼ぶ。
    """
    fig.subplots_adjust(right=frac)


def save_fig(fig, stem, out=None):
    """図を SVG で保存してパスを表示する。

    stem は拡張子なしの名前（例 'Step1_period_balance'）。
    点が数千個ある散布図は，散布図だけ rasterized=True にしておくと
    軸と文字はベクタのままファイルが軽くなる。
    """
    d = Path(out) if out else OUT
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.{FIG_EXT}'
    fig.savefig(path, format=FIG_EXT, dpi=RASTER_DPI, bbox_inches='tight')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB)')
    return path


# ---------------------------------------------------------------------------
# 対話的な図（SVG はそのまま残す）
# ---------------------------------------------------------------------------
# 散布図の点が何百個あると，注記を付けられるのはごく一部である。残りの点は
# 「どの語か」が分からないまま眺めることになる。かといって全点に名前を
# 付ければ図は読めない。
#
# そこで**同じ図から2つ出す**。
#   * ``<stem>.svg``  … 論文・配布用。これまでどおり。加筆も拡大も自由
#   * ``<stem>.html`` … 授業・探索用。SVG をそのまま埋め込み，
#                       その上に当たり判定を重ねて，指した点の語を出す
#
# **HTML は SVG を作り直さない。同じ SVG を中に入れる。** 別に描き直すと
# 図が2種類できて，どちらが正かが分からなくなる。注記（bursty な語の
# ラベル）も SVG の中にあるのでそのまま残る。
#
# 外部の JS ライブラリは使わない。CDN が塞がれたマシンでも開けるようにする。
INTERACTIVE_CSS = """
:root { --ink:#1f1e1b; --ink2:#5a5a55; --line:#d8d7d0; --surface:#ffffff;
        --wash:#f7f7f4; --accent:#184f95; }
* { box-sizing:border-box; }
body { margin:0; padding:24px 16px 48px; background:var(--wash);
       color:var(--ink); font-family:"Hiragino Sans","Noto Sans JP",
       "Yu Gothic",system-ui,sans-serif; line-height:1.6; }
.wrap { max-width:1100px; margin:0 auto; }
h1 { font-size:1.15rem; margin:0 0 .2em; font-weight:650; }
.sub { color:var(--ink2); font-size:.86rem; margin:0 0 1.1em; }
.card { background:var(--surface); border:1px solid var(--line);
        border-radius:10px; padding:14px; }
.figbox { position:relative; }
.figbox svg { width:100%; height:auto; display:block; }
#hit { position:absolute; inset:0; cursor:crosshair; }
#ring { position:absolute; width:22px; height:22px; margin:-11px 0 0 -11px;
        border:2px solid var(--accent); border-radius:50%;
        pointer-events:none; opacity:0; transition:opacity .08s; }
#tip { position:absolute; z-index:5; min-width:190px; max-width:290px;
       background:var(--surface); border:1px solid var(--line);
       border-radius:8px; box-shadow:0 6px 20px rgba(0,0,0,.13);
       padding:9px 11px; font-size:.8rem; pointer-events:none; opacity:0;
       transition:opacity .08s; }
#tip .term { font-size:1.05rem; font-weight:650; letter-spacing:.02em;
             margin-bottom:.35em; word-break:break-all; }
#tip dl { display:grid; grid-template-columns:auto 1fr; gap:1px 10px;
          margin:0; }
#tip dt { color:var(--ink2); font-size:.74rem; white-space:nowrap; }
#tip dd { margin:0; text-align:right; font-variant-numeric:tabular-nums;
          font-weight:600; }
.bar { display:flex; gap:10px; align-items:center; flex-wrap:wrap;
       margin:14px 0 0; font-size:.82rem; color:var(--ink2); }
.bar input { font:inherit; padding:5px 9px; border:1px solid var(--line);
             border-radius:6px; min-width:190px; background:var(--surface); }
.bar a { color:var(--accent); }
table { border-collapse:collapse; width:100%; font-size:.78rem;
        margin-top:10px; }
th,td { padding:4px 8px; border-bottom:1px solid #ecebe6; text-align:left;
        white-space:nowrap; }
th { background:var(--wash); position:sticky; top:0; font-weight:650; }
td.num { text-align:right; font-variant-numeric:tabular-nums; }
tbody tr:hover td { background:var(--wash); }
tbody tr.on td { background:#eaf1fb; }
.scroll { max-height:340px; overflow:auto; border:1px solid var(--line);
          border-radius:8px; margin-top:10px; }
.hint { font-size:.78rem; color:var(--ink2); margin:.6em 0 0; }
#links { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#links line { stroke:var(--accent); stroke-width:1.1; opacity:.55; }
#links circle { fill:none; stroke:var(--accent); stroke-width:1.4; opacity:.8; }
#marks { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#marks circle { fill:none; stroke:#d55e00; stroke-width:1.6; opacity:.9; }
#tip .notes { margin:.45em 0 0; font-size:.76rem; color:var(--ink);
              border-top:1px solid var(--line); padding-top:.4em;
              line-height:1.5; word-break:break-all; }
#tip .notes b { color:var(--ink2); font-weight:600; }
.prov { font-size:.72rem; color:var(--ink2); margin:.9em 0 0;
        border-top:1px solid var(--line); padding-top:.6em;
        font-variant-numeric:tabular-nums; }
.danger { background:#fdf0ea; border:1px solid #e8a37c; border-radius:8px;
          padding:9px 12px; font-size:.85rem; color:#8a3b10;
          margin:0 0 12px; }
"""

INTERACTIVE_JS = r"""
// 点は data-* ではなく JSON で渡す。語はコーパス由来の任意の文字列なので，
// **HTML に文字列連結で差し込まない**（textContent で入れる）。
// 見出し（keys・nhead）は全点で同じなら1回だけ入っている。点が1万個ある
// 図では，これで HTML が 1 MB 以上軽くなる。古い形（配列だけ）も読む。
const RAW = JSON.parse(document.getElementById('pts-data').textContent);
const PTS = Array.isArray(RAW) ? RAW : RAW.pts;
const KEYS = (RAW && RAW.keys) || [];
const NHEAD = (RAW && RAW.nhead) || '';
const LINKNOTES = !!(RAW && RAW.linknotes);
function pairsOf(p) {
  if (p.fields) return p.fields;
  if (p.v) return p.v.map((x, i) => [KEYS[i] || '', x]);
  return [];
}
function notesOf(p) {
  if (p.notes) return p.notes;
  if (p.n) return [NHEAD, p.n];
  // 本文が無く linknotes が立っているときは，線で結ぶ先の語を並べる
  if (LINKNOTES && p.links && p.links.length) {
    return [NHEAD, p.links.map(j => (PTS[j] || {}).term || '').join(' ')];
  }
  return null;
}
const box = document.getElementById('hit');
const tip = document.getElementById('tip');
const ring = document.getElementById('ring');
const rows = Array.from(document.querySelectorAll('tbody tr'));
const links = document.getElementById('links');
const marks = document.getElementById('marks');

// **最も近い点を拾う。** 点の直径は数ピクセルしかないので，
// 「真上に置く」ことを要求すると誰も当てられない（dataviz の規則）。
// カーソルに最も近い点を選び，遠すぎるときだけ何も出さない。
function nearest(px, py, w, h) {
  let best = null, bd = 1e9;
  for (const p of PTS) {
    const dx = p.x * w - px, dy = p.y * h - py;
    const d = dx * dx + dy * dy;
    if (d < bd) { bd = d; best = p; }
  }
  return Math.sqrt(bd) <= 34 ? best : null;   // 34px より遠ければ出さない
}

function fill(p) {
  tip.textContent = '';
  const h = document.createElement('div');
  h.className = 'term';
  h.textContent = p.term;                     // ← 連結しない
  tip.appendChild(h);
  const dl = document.createElement('dl');
  for (const [k, v] of pairsOf(p)) {
    const dt = document.createElement('dt'); dt.textContent = k;
    const dd = document.createElement('dd'); dd.textContent = v;
    dl.appendChild(dt); dl.appendChild(dd);
  }
  tip.appendChild(dl);
  const nt = notesOf(p);
  if (nt) {                                   // 近傍語など，横に長い情報
    const n = document.createElement('p');
    n.className = 'notes';
    const b = document.createElement('b');
    b.textContent = nt[0] + ' ';
    n.appendChild(b);
    n.appendChild(document.createTextNode(nt[1]));
    tip.appendChild(n);
  }
}

// **原空間での近傍を線で結ぶ。** 画面の近さは射影の結果にすぎない。
// 線が遠くへ伸びるなら，その点の近傍関係は2次元に収まっていない。
// これを見せるのが，この図でいちばん大事なところである。
function drawLinks(p, w, h) {
  if (!links) return;
  while (links.firstChild) links.removeChild(links.firstChild);
  if (!p.links || !p.links.length) return;
  const NS = 'http://www.w3.org/2000/svg';
  for (const j of p.links) {
    const q = PTS[j];
    if (!q) continue;
    const ln = document.createElementNS(NS, 'line');
    ln.setAttribute('x1', p.x * w); ln.setAttribute('y1', p.y * h);
    ln.setAttribute('x2', q.x * w); ln.setAttribute('y2', q.y * h);
    links.appendChild(ln);
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', q.x * w); c.setAttribute('cy', q.y * h);
    c.setAttribute('r', 5);
    links.appendChild(c);
  }
}

let cur = null;
function show(p, px, py) {
  const w = box.clientWidth, h = box.clientHeight;
  if (p !== cur) { fill(p); drawLinks(p, w, h); cur = p; }
  ring.style.left = (p.x * w) + 'px';
  ring.style.top = (p.y * h) + 'px';
  ring.style.opacity = 1;
  tip.style.opacity = 1;
  // はみ出さないように寄せる
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lx = px + 16, ly = py + 14;
  if (lx + tw > w) lx = px - tw - 16;
  if (ly + th > h) ly = py - th - 14;
  tip.style.left = Math.max(0, lx) + 'px';
  tip.style.top = Math.max(0, ly) + 'px';
  rows.forEach(r => r.classList.toggle('on', r.dataset.i === String(p.r)));
}
function hide() {
  tip.style.opacity = 0; ring.style.opacity = 0; cur = null;
  if (links) while (links.firstChild) links.removeChild(links.firstChild);
  rows.forEach(r => r.classList.remove('on'));
}

box.addEventListener('pointermove', e => {
  const r = box.getBoundingClientRect();
  const p = nearest(e.clientX - r.left, e.clientY - r.top, r.width, r.height);
  if (p) show(p, e.clientX - r.left, e.clientY - r.top); else hide();
});
box.addEventListener('pointerleave', hide);

// 表の行にカーソルを乗せても，図の上の点が光る（逆引き）。
// **カーソルが使えない人にも同じ情報が届くように**，表を必ず添える
// （点が数千を超える図だけは表を絞る。絞ったことは図の下に明記する）。
rows.forEach(r => {
  r.addEventListener('mouseenter', () => {
    // 表の行は論理点。図の上では**先頭の面**の点を光らせる
    const p = PTS[Number(r.dataset.i)];
    if (!p) return;
    const w = box.clientWidth, h = box.clientHeight;
    show(p, p.x * w, p.y * h);
  });
  r.addEventListener('mouseleave', hide);
});

// 絞り込み。語・作品・時代のどれでも当たる
const q = document.getElementById('q');
if (q) q.addEventListener('input', () => {
  const s = q.value.trim();
  let n = 0;
  rows.forEach(r => {
    const hit = !s || r.textContent.includes(s);
    r.style.display = hit ? '' : 'none';
    if (hit) n++;
  });
  // 表を絞った図では，**表に無い語も図の上では当たる**。
  // 表の件数だけを出すと「無い」と誤解されるので両方を出す。
  const nlog = Number(document.body.dataset.nlog || rows.length);
  let extra = '';
  if (s && rows.length < nlog) {
    const seen = new Set();
    for (const p of PTS) if (p.term.includes(s)) seen.add(p.r);
    extra = '（図の上 ' + seen.size + ' 件）';
  }
  document.getElementById('count').textContent = n + ' 件' + extra;
  // **図の上にもマーカーを付ける。** 表だけ絞っても「どこにあるか」は分からない。
  if (!marks) return;
  while (marks.firstChild) marks.removeChild(marks.firstChild);
  if (!s) return;
  const NS = 'http://www.w3.org/2000/svg';
  const w = box.clientWidth, h = box.clientHeight;
  let drawn = 0;
  for (const p of PTS) {
    if (!p.term.includes(s)) continue;
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', p.x * w); c.setAttribute('cy', p.y * h);
    c.setAttribute('r', 7);
    marks.appendChild(c);
    if (++drawn > 400) break;          // マーカーが多すぎると図が読めない
  }
});
"""


def save_interactive(fig, ax, stem, xs, ys, tips, out=None, title='',
                     note='', table_cols=None, source=None, id_col='語',
                     hint='', table_idx=None, coords=None):
    """SVG を保存し，**同じ SVG を埋め込んだ対話的な HTML** も書く。

    ``xs`` ``ys`` はデータ座標，``tips`` は点ごとの情報
    （``{'term': 語, 'fields': [(見出し, 値), …]}`` の並び）。
    3つの長さは一致していなければならない。ずれたまま描くと，
    **指した点と出る語が食い違う**（注記の添字ずれと同じ事故）。

    位置は「図全体に対する割合」で書き出す。SVG を ``width:100%`` で
    伸縮させても割合は変わらないので，どんな幅でも点と当たり判定が
    合う。座標は matplotlib の変換を通して得るので，**図と HTML で
    座標の計算が二重にならない**。

    ``source`` に入力ファイルのパスを渡すこと。**どの表から描いた図かを
    HTML の末尾に刻む。** これが無いと，試験用の作りかけのデータから
    描いた図と，本番のデータから描いた図が見分けられない。
    入力がプロジェクトの外（``/tmp`` など）にあるときは
    「試験用」と赤字で出し，配布してはいけないことを図自身に言わせる。

    ``id_col`` は表の第1列の見出し（既定「語」。作品を点にする図では
    「作品」などに変える）。

    ``ax`` には**面の並び**も渡せる（``[axes[0], axes[1]]``）。同じ点を
    別の色分けで2面に描いた図では，どちらの面を指しても同じ情報が出る。
    表の行は点ごとに1行だけ作る（面の数だけ重複させない）。

    ``table_idx`` は**表に載せる点の添字**（既定は全点）。点が数千を超える
    図では表を全件出すと HTML が数 MB になり，読む側にも役に立たない。
    そのときは載せる点を選ぶ。**ただし図の当たり判定と検索は全点に効く**
    ので，表に無い語も指せるし検索で図にマーカーが付く。表を絞ったときは，
    何件のうち何件を載せたかを HTML に明記する（黙って捨てないこと）。

    ``coords`` は**面ごとの座標**（``[(x1, y1), (x2, y2)]``）。同じ点を
    **違う座標系**で2面に描いた図（主成分分析と UMAP の比較など）で使う。
    渡さなければ全部の面で ``xs`` ``ys`` を使う。
    ⚠ 面ごとに座標が違うのに ``coords`` を渡さないと，2面めの当たり判定が
    1面めの座標で置かれる。**図は出るが，指した点と出る語が食い違う。**
    """
    import json as _json
    if not (len(xs) == len(ys) == len(tips)):
        raise ValueError(
            f'save_interactive: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'情報={len(tips)}）。座標と情報を同じ添字で絞り込むこと。')

    svg_path = save_fig(fig, stem, out=out)
    outdir = svg_path.parent

    # ---- 埋め込む SVG は**切り取らずに**保存する ------------------------
    # save_fig は bbox_inches='tight' で余白を詰めるため，図全体に対する
    # 割合と，ファイルの座標系がずれる。埋め込み用は詰めずに出す。
    import io
    buf = io.StringIO()
    fig.savefig(buf, format='svg', dpi=RASTER_DPI)
    svg = buf.getvalue()
    svg = svg[svg.index('<svg'):]          # XML 宣言と DOCTYPE を落とす

    # ---- 点の位置を図全体に対する割合で得る -----------------------------
    axes_list = list(ax) if isinstance(ax, (list, tuple, np.ndarray)) else [ax]
    W, H = fig.bbox.width, fig.bbox.height
    n_pts = len(tips)

    # **見出しは点ごとに書かない。** 点が1万個ある図では，
    # 「品詞」「頻度」…という見出しを1万回繰り返すだけで HTML が
    # 1 MB 以上ふくらむ。全点で見出しが同じなら1回だけ書き，
    # 値の並びだけを点に持たせる（JS 側で組み直す）。
    keys = [str(k) for k, _ in tips[0].get('fields', [])] if tips else []
    same_keys = bool(keys) and all(
        [str(k) for k, _ in t.get('fields', [])] == keys for t in tips)
    nheads = {str(t['notes'][0]) for t in tips if t.get('notes')}
    nhead = next(iter(nheads)) if len(nheads) == 1 else ''
    # 本文を渡さず ``notes=(見出し, None)`` としたときは，
    # ``links`` の先の語を JS 側で並べる
    linknotes = bool(nhead) and any(
        t.get('notes') and t['notes'][1] is None and t.get('links')
        for t in tips)

    pts = []
    if coords is not None and len(coords) != len(axes_list):
        raise ValueError(
            f'save_interactive: coords の数が面の数と違う'
            f'（面 {len(axes_list)} / coords {len(coords)}）')
    for k, axk in enumerate(axes_list):
      xk, yk = (coords[k] if coords is not None else (xs, ys))
      if not (len(xk) == len(yk) == n_pts):
          raise ValueError(
              f'save_interactive: 面 {k} の座標の数が情報の数と違う'
              f'（x={len(xk)}, y={len(yk)}, 情報={n_pts}）')
      pxy = axk.transData.transform(np.column_stack([np.asarray(xk, float),
                                                     np.asarray(yk, float)]))
      for j, (t, (px, py)) in enumerate(zip(tips, pxy)):
        i = k * n_pts + j
        # **変数名に注意。** ここを d と書くと，上で取った出力先 d
        # （svg_path.parent）を上書きして，最後に d / '....html' が
        # 「dict ÷ str」になる。実際に踏んだ。名前は使い回さない。
        # 添字 i は JS では使わない（行は r で引く）。点が1万個ある図では
        # 使わない値も 100 KB 単位で効くので書かない。
        rec = {'r': j, 'term': str(t.get('term', '')),
               'x': round(float(px) / W, 6),
               'y': round(1 - float(py) / H, 6)}        # SVG は上が 0
        if same_keys:
            rec['v'] = [str(b) for _, b in t.get('fields', [])]
        else:
            rec['fields'] = [[str(a), str(b)] for a, b in t.get('fields', [])]
        if t.get('notes'):
            # ('見出し', '本文') の2つ組。横に長い情報（近傍語など）。
            # 本文を None にすると，**線で結ぶ先の語を JS が並べる**
            # （同じ語の列を点ごとに書かずに済む。1万点で 1 MB 近く効く）
            if t['notes'][1] is None:
                pass
            elif nhead:
                rec['n'] = str(t['notes'][1])
            else:
                rec['notes'] = [str(t['notes'][0]), str(t['notes'][1])]
        if t.get('links'):
            # 原空間での近傍の添字。**同じ面の中で**線を結ぶ
            rec['links'] = [k * n_pts + int(q) for q in t['links']]
        pts.append(rec)

    cols = table_cols or keys
    head = f'<tr><th>{_esc(id_col)}</th>' + ''.join(
        f'<th>{_esc(c)}</th>' for c in cols) + '</tr>'
    # 数字の列だけ右寄せにする。時代名や作品 ID を右寄せにすると読みにくい。
    def _numish(v):
        t = str(v).strip().replace('%', '').replace(',', '')
        t = t.lstrip('+-')
        return bool(t) and t.replace('.', '', 1).isdigit()

    # 表に載せる点。**面の数だけ重複させない**（論理点1つに1行）
    if table_idx is None:
        order = list(range(n_pts))
    else:
        seen, order = set(), []
        for i in [int(q) for q in table_idx]:   # 重複を除きつつ順序は保つ
            if 0 <= i < n_pts and i not in seen:
                seen.add(i); order.append(i)

    # 表は **tips から作る**（点の JSON は見出しを省いてあるので）
    body = []
    for j in order:
        fv = {str(a): str(b) for a, b in tips[j].get('fields', [])}
        tds = ''
        for c in cols:
            v = fv.get(c, '')
            cls = ' class="num"' if _numish(v) else ''
            tds += f'<td{cls}>{_esc(v)}</td>'
        body.append(f'<tr data-i="{j}">'
                    f'<td>{_esc(tips[j].get("term", ""))}</td>{tds}</tr>')

    # ---- 由来を図自身に刻む -------------------------------------------
    # **どの表から描いた図かが分からないと，試験用のデータで描いた図が
    # 本物として配られる。** 実際に起きた（2026-09-22）。
    import datetime as _dt
    stamp = _dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')
    src = Path(source) if source else None
    prov = f'点 {len(pts)} 個／作図 {stamp}'
    warn = ''
    if src is not None:
        try:
            mt = _dt.datetime.fromtimestamp(src.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        except OSError:
            mt = '不明'
        prov = f'入力 {src.name}（更新 {mt}）／' + prov
        # プロジェクトの外（/tmp など）から描いた図は試験用である
        try:
            outside = not str(src.resolve()).startswith(str(ROOT.resolve()))
        except Exception:                               # noqa: BLE001
            outside = True
        if outside or '/tmp/' in str(src):
            warn = ('<p class="danger">⚠ <b>試験用の入力から作った図である。'
                    f'配布してはいけない。</b>（入力 {_esc(str(src))}）</p>')
            prov = f'入力 {_esc(str(src))}／' + f'点 {len(pts)} 個／作図 {stamp}'

    html = f"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title or stem)}</title>
<style>{INTERACTIVE_CSS}</style></head>
<body data-nlog="{n_pts}" data-ntable="{len(body)}"><div class="wrap">
<h1>{_esc(title or stem)}</h1>
<p class="sub">{_rich(note)}</p>
{warn}
<div class="card">
  <div class="figbox">
    {svg}
    <svg id="links"></svg><svg id="marks"></svg>
    <div id="hit"></div><div id="ring"></div><div id="tip"></div>
  </div>
  <p class="hint">{_rich(hint or '点にカーソルを近づけると語が出る（最も近い点を拾うので，真上に置かなくてよい）。図の中の注記は静止版と同じものである。')}</p>
  <div class="bar">
    <input id="q" type="search" placeholder="語・作品・時代で絞り込む">
    <span id="count">{len(body)} 件</span>
    <span>·</span>
    {(f'<span>表は {len(body)} 件（図の点は {n_pts} 件。'
      '表に無い語も図の上で指せる。検索は図のマーカーにも効く）</span><span>·</span>')
     if len(body) < n_pts else ''}
    <a href="{_esc(svg_path.name)}" download>SVG を保存</a>
    <span>（この HTML の中の図はその SVG そのもの）</span>
  </div>
  <div class="scroll"><table><thead>{head}</thead>
    <tbody>{''.join(body)}</tbody></table></div>
  <p class="prov">{prov}</p>
</div>
<script type="application/json" id="pts-data">{_json.dumps(
    {'keys': keys if same_keys else [], 'nhead': nhead,
     'linknotes': linknotes, 'pts': pts},
    ensure_ascii=False, separators=(',', ':'))}</script>
<script>{INTERACTIVE_JS}</script>
</div></body></html>
"""
    path = outdir / f'{stem}.html'
    path.write_text(html, encoding='utf-8')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB・対話版／'
          f'点 {len(pts)} 個)')
    return svg_path, path


def _esc(s):
    """HTML の特殊文字を落とす。**語はコーパス由来なので必ず通す。**"""
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _rich(s):
    """注記の ``**…**`` だけを太字にする。

    説明文をノートブックと同じ書き方（Markdown 風）で書けるようにする。
    **先に必ず _esc を通す**ので，タグを書き込まれる余地は無い。
    ``**`` のままだと HTML では記号がそのまま出て読みにくい。
    """
    import re as _re
    return _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', _esc(s))


PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
           '#56B4E9', '#D55E00', '#F0E442', '#666666']

# --- 順序のあるものを色分けするための1色相のランプ -----------------------------
# **時代・年次・段階のように順序のあるものを，上の8色で色分けしてはいけない。**
# 明治中期が青で明治後期が黄なら，隣り合う時代が隣り合う色にならず，
# 「時代が下るとどちらへ動くか」という肝心のことが読めなくなる。
# 1色相の濃淡にすれば，近いもの同士が近い色になり，勾配がそのまま見える。
# 散布図の点は白地の上に置くので，いちばん明るい段は 100 ではなく
# 250（背景との対比 2:1）から始める。100 は面を色分けするとき（ヒートマップ）用。
SEQ_BLUE_STEPS = ['#86b6ef', '#5598e7', '#3987e5',
                  '#256abf', '#184f95', '#0d366b']
SEQ_BLUE = LinearSegmentedColormap.from_list('jlit_blue', SEQ_BLUE_STEPS)

# 離散の順序（4区分など）を色分けするときはこちら。隣の段と明度差が十分あり，
# いちばん明るい段も背景から浮く（対比 2:1 以上）ことを確かめてある。
SEQ_BLUE_5 = ['#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b']

# 大分類の2色。散布図ではどの2点も隣り合いうるので**全ペアが
# 見分けられる必要**があり，使える色数は多くない。2色に絞って，
# 下位の区別はマーカーの形に持たせる。
GENRE_C = {'Fiction': '#2a78d6', 'Nonfiction': '#eb6834'}

# --- 初出年の5段 ---------------------------------------------------------
# 切れ目は period と同じ 1900／1912／1926／1945。
# **6段にはできない。** 1色相の濃淡で順序を見せるには，隣り合う段の明度差が
# 0.06 以上要る。この青系ランプは 250→700 で明度差にして 0.30 ほどしか幅が
# 無いので，段を6つ取るとどこかが 0.05 台に落ち，隣が見分けられなくなる。
# そこで作品数3点の明治前期（〜1886）を明治中期にまとめて5段とする。
YEAR_EDGES = [1900, 1912, 1926, 1945]
YEAR_LABELS = ['〜1899 明治前・中期', '1900-1911 明治後期', '1912-1925 大正',
               '1926-1944 昭和戦前', '1945- 昭和戦後']


def year_bands(years):
    """初出年を5段に畳み，``(段番号, ラベル, 色)`` を返す。

    段番号は 0〜4。**初出年が読めないものは -1** にする。0 に落とすと
    年の分からない作品が全部いちばん古い段に入り，通時の議論が崩れる。

    時代で色分けする図はすべてこれを通すこと。同じ色が全ステップで同じ時代を
    指すようになり，Step 1 の図と Step 7 の図を並べて読める。
    """
    y = pd.to_numeric(pd.Series(list(years)), errors='coerce')
    code = np.full(len(y), -1, dtype=int)
    ok = y.notna().values
    if ok.any():
        code[ok] = np.digitize(y[ok].values, YEAR_EDGES)
    return code, YEAR_LABELS, SEQ_BLUE_5


def run_script(script, *args, tail=4000):
    """scripts/ のスクリプトを実行し，標準出力・標準エラー・終了コードを必ず表示する。

    print(r.stdout or r.stderr) では，標準出力が空でないときに
    エラーの内容が隠れてしまう。学習用には両方見えるほうがよい。
    """
    cmd = [sys.executable, str(ROOT / 'scripts' / script)] + [str(a) for a in args]
    print('$ python', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, capture_output=True, text=True)

    def _tail(s):
        # 末尾 tail 文字だけを出す。行の途中で切らないよう，切ったときは
        # 次の改行から始め，前を省いたことを明示する
        if len(s) <= tail:
            return s
        s = s[-tail:]
        return '（…前略）\n' + s[s.find('\n') + 1:]

    if r.stdout:
        print(_tail(r.stdout))
    if r.stderr.strip():
        # 標準エラーには**エラー以外**も出る。MALLET は学習の進み具合
        # （<10> LL/token: …）と途中のトピック上位語をここに書く。
        # 判断は [exit 0] かどうかで行う
        print('--- stderr（進行ログを含む。エラーとは限らない）---')
        print(_tail(r.stderr))
    print(f'[exit {r.returncode}]' + ('' if r.returncode == 0 else '  ← 0 でなければ失敗'))
    return r


# 使うメタデータ。増補分（45点）を含む v3 があればそちらを優先する。
# v2 は v1 の 64 点しか無いので，増補後のコーパスで v2 を使うと
# 突合が外れて period も genre も空になる（Step 3 で v3 を作る）。
# Step 3 で自分が作った v3 は *_local.csv に書かれる（配布版は上書きしない。
# 上書きすると git pull のたびに衝突する）。自分の版 > 配布版 v3 > v2 の順。
for _m in ('corpus_metadata_v3_local.csv', 'corpus_metadata_v3.csv',
           'corpus_metadata_v2.csv'):
    META = ROOT / 'metadata' / _m
    if META.exists():
        break

# 図・表の書き出し先は自分の作業フォルダ my_work/results/。my_work/ は
# コースのリポジトリの外扱い（.gitignore）で，自分の GitHub にバックアップを取る。
OUT = ROOT / 'my_work' / 'results'
OUT.mkdir(parents=True, exist_ok=True)
print('OUT  =', OUT)


In [ ]:
TOK = ROOT/'data'/'tokens'
SRC = TOK/'tokens_lemma'
if not SRC.exists():
    print('Step 3 の出力がありません。')
else:
    run_script('07_descriptive_stats.py', '--tokens', SRC, '--tsv', TOK/'tsv',
               '--meta', META,
               '--out', OUT/'descriptive', '--mfw', 300)

## 1. 語彙の豊かさ — TTR の罠

Type-Token Ratio（異なり語数 ÷ 延べ語数）は直感的だが，
**標本サイズに強く依存する**。長い作品ほど TTR は必ず下がる。

v1 の実測がその典型である。

| 作品 | 語数 | TTR×1000 |
|---|---:|---:|
| 海野十三『敗戦日記』 | 50,131 | 169.5 |
| 藤村『夜明け前』 | 502,937 | **45.2** |

『夜明け前』の語彙が貧しいのではない。長いだけである。
標本サイズに頑健な指標を使う。

- **Guiraud's R** = V / √N
- **Yule's K**（頻度分布の集中度。長さにほぼ依存しない）
- **エントロピー**（bit）

In [ ]:
p = OUT/'descriptive'/'work_profile.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    wp = pd.read_csv(p)
    fig, axes = plt.subplots(1,3, figsize=(14,4))
    for ax, y, lab in zip(axes, ['ttr','guiraud_R','yule_K'],
                          ['TTR','Guiraud R','Yule K']):
        ax.scatter(wp.tokens, wp[y], s=30, color=PALETTE[0], alpha=.8)
        ax.set_xscale('log'); ax.set_xlabel('延べ語数（対数）'); ax.set_ylabel(lab)
        rho = np.corrcoef(np.log(wp.tokens), wp[y])[0,1]
        ax.set_title(f'{lab}  (r={rho:+.2f})')
        ax.spines[['top','right']].set_visible(False)
    fig.tight_layout(); save_fig(fig, 'Step4_lexical_richness'); plt.show()
    print('→ 相関の絶対値が小さい指標ほど，長さに頑健である。')

## 2. Burrows's Delta を手で計算する

Delta の手順は三つしかない。

1. 最頻語 N 語の**相対頻度**を求める（ここでは 1 万語あたり）
2. 語ごとに **z スコア**にする（頻度の大小によらず各語を等しく扱う）
3. 2 つのテクストの z の**差の絶対値を，語について平均**する——これが距離

単純さが強みで，少ない訓練データでもよく効く。ただしノートブックでは
`np.abs(Z[:,None]-Z[None]).mean(2)` の1行になってしまい，**何をしているかが
見えない**。そこで二段で進める。

| | どこで | 規模 | 目的 |
|---|---|---|---|
| **2.1** | Excel | 最頻語 50 語 × 9 作品 | 1 手順＝1 シート。セルと数式で計算の中身を見る |
| **2.2** | ノートブック | 同じ 50 語 × 9 作品 | 同じ数値を pandas で出し，**Excel と一致する**ことを確かめる |
| **2.3** | — | — | 作品X の正体と，結果の読み方 |
| **2.4** | ノートブック | 300 語 × 全作品 | Excel では扱えない規模に広げる |

作家を伏せた「作品X」1 点と，既知の 4 作家（各 2 点）を比べる。
既知の作家には，作品X と同時代の作家と，時代も読者層も離れた作家とが
混ざっている。**遠い理由が作家の違いだけとは限らない**ことにも注意する。
**作品X が誰の作品かは 2.3 まで見ないこと。**

### 2.1 Excel で手計算する

次のセルが `my_work/results/Step4_delta_manual.xlsx` を作る。Excel（Numbers・
LibreOffice でもよい）で開き，`説明` シートから順に見ていく。

- **最頻語・平均・標準偏差は既知の作品だけから決める。** 作品X に物差しを
  動かさせないためである（Burrows 2002）
- 標準偏差は**母標準偏差**（Excel の `STDEVP`，numpy の `ddof=0`）。
  `07_descriptive_stats.py` も同じ
- シートを**並べ替えないこと**（数式の参照が崩れる）

In [ ]:
# ---- 手計算用のブックを作る ------------------------------------------
# 作品を変えるときは '--questioned 作家:題' '--known 作家:題,作家:題,…' を足す
DX = OUT/'Step4_delta_manual.xlsx'
run_script('17_delta_workbook.py', '--tokens', SRC, '--meta', META,
           '--mfw', 50, '--out', DX)

#### 演習 0 — Excel の上で（15 分）

1. `6_Delta` で，作品X に**いちばん近い作品**と**いちばん近い作家プロファイル**を
   書き留める。両者は一致するか
2. `5_z差` で色の濃い（差の大きい）語を 3 つ挙げる。それは作家の癖か，
   **語りの人称や登場人物名**など別の要因か
3. `7_語数Nを変える` の黄色のセルに 10・20・50 を入れ，順位の変わり方を見る
4. `1_度数` の数字を 1 つ書き換え，影響がどのシートまで伝わるかを追う
   （試したら元に戻すか，上のセルでブックを作り直す）

### 2.2 同じ計算をノートブックで

Excel の `1_度数` から**入力値だけ**を読み，同じ手順を pandas で1行ずつ書く。
変数名はシート名に合わせてある。最後に Excel の結果と突き合わせる。

In [ ]:
# ---- Excel と同じ計算を pandas で ------------------------------------
import json, openpyxl
if need(DX, '上のセルでブックを作ること'):
    ws = openpyxl.load_workbook(DX)['1_度数']
    R_AU, R_TI, R_N, R0 = 3, 4, 5, 6          # ブックの配置（17_delta_workbook.py と同じ）
    ncol = ws.max_column
    names   = [ws.cell(R_TI, c).value for c in range(3, ncol + 1)]
    authors = [ws.cell(R_AU, c).value for c in range(3, ncol + 1)]
    words, rows = [], []
    r = R0
    while ws.cell(r, 2).value is not None:
        words.append(ws.cell(r, 2).value)
        rows.append([ws.cell(r, c).value for c in range(3, ncol + 1)])
        r += 1
    counts = pd.DataFrame(rows, index=words, columns=names)            # 1_度数
    totals = pd.Series([ws.cell(R_N, c).value for c in range(3, ncol + 1)], index=names)
    X = names[-1]                                                      # 作品X
    known = names[:-1]
    author_of = dict(zip(known, authors[:-1]))

    rel  = counts / totals * 10000                                     # 2_相対頻度
    mu   = rel[known].mean(axis=1)                                     # 3_平均と標準偏差
    sd   = rel[known].std(axis=1, ddof=0)                              #   STDEVP と同じ
    z    = rel.sub(mu, axis=0).div(sd, axis=0)                         # 4_zスコア
    diff = z[known].rsub(z[X], axis=0).abs()                           # 5_z差
    delta = diff.mean()                                                # 6_Delta

    prof = z[known].T.groupby(author_of).mean().T                      # 作家プロファイル
    delta_a = prof.rsub(z[X], axis=0).abs().mean()

    t = pd.DataFrame({'作家': [author_of[k] for k in known], '作品': known,
                      'Delta': delta.values}).sort_values('Delta')
    t.insert(0, '順位', range(1, len(t) + 1))
    show(t, caption=f'{X} との Delta（作品ごと・{len(words)} 語）', fmt={'Delta': '{:.4f}'})
    ta = delta_a.sort_values().rename('Delta').reset_index().rename(columns={'index': '作家'})
    ta.insert(0, '順位', range(1, len(ta) + 1))
    show(ta, caption=f'{X} との Delta（作家プロファイル）', fmt={'Delta': '{:.4f}'})

    # ---- Excel と突き合わせる ---------------------------------------------
    # Excel で開いて「保存」すると計算結果がファイルに残るので，それを読む。
    # 保存していなければ値が無い（数式しか入っていない）。
    ws6 = openpyxl.load_workbook(DX, data_only=True)['6_Delta']
    xl = {ws6.cell(r, 3).value: ws6.cell(r, 4).value
          for r in range(5, 5 + len(known)) if ws6.cell(r, 1).value == '作品'}
    if all(v is None for v in xl.values()):
        print('[info ] ブックに計算結果が保存されていない。Excel で開いて保存してから'
              'このセルを実行し直すと，自動で突き合わせる。')
        print('        それまでは 6_Delta の D 列と上の表を目で比べること。')
    else:
        gap = max(abs(xl[k] - delta[k]) for k in known if xl.get(k) is not None)
        print(f'[{"ok  " if gap < 1e-6 else "NG  "}] Excel との差の最大 = {gap:.2e}'
              + ('（一致）' if gap < 1e-6 else ' — 1_度数 を書き換えたままではないか'))

### 2.3 作品X の正体と，結果の読み方

次のセルで作品X を明かす。**演習 0 の答えを書き留めてから実行すること。**

In [ ]:
exp = json.loads((DX.parent / (DX.stem + '_expected.json')).read_text(encoding='utf-8'))
qx = exp['questioned']
print(f"作品X ＝ {qx['author']}『{qx['title']}』")
best_w = min(exp['delta_work'], key=exp['delta_work'].get)
best_a = min(exp['delta_author'], key=exp['delta_author'].get)
print(f'いちばん近い作品          : {best_w}')
print(f'いちばん近い作家プロファイル: {best_a}'
      + ('  ← 正解' if best_a == qx['author'] else '  ← 外れ'))

**読み方の要点**

- **作品単位と作家単位で答えが変わりうる。** 同じ作家でも作品ごとに
  語りの人称（一人称の「私」）や文体が違う。1 点ずつ比べると，たまたま
  似た**別の作家の作品**が割り込む。作家の2作品の z を平均した**プロファイル**
  は，作品ごとの揺れをならす
- **近さの理由を必ず語で確かめる。** `5_z差` で差を作っている語が，
  登場人物名（最頻語に固有名詞が紛れ込む）や「私」のような人称なら，
  Delta は作家ではなく**語りの形式**を測っている
- **N を変えると順位が動く。** 10 語では少数の語に振り回される。
  報告には必ず N を書き，いくつかの N で結論が変わらないことを示す
- **候補に真の作者がいなくても，いちばん近い誰かは必ず出る。**
  Delta は「この中で誰に近いか」しか答えない

### 2.4 全作品・300 語に広げる

ここからは Excel では扱えない規模である。手順は 2.2 とまったく同じで，
作品 × 作品のすべての組について Delta を出す（全作品で z を取るので，
2.1–2.2 のような「既知／問題」の区別はない）。

In [ ]:
p = OUT/'descriptive'/'freq_matrix_mfw.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    F = pd.read_csv(p, index_col=0)
    # ddof=0（母標準偏差）。07_descriptive_stats.py・Excel の STDEVP と揃える
    Z = (F - F.mean()) / F.std(ddof=0).replace(0, 1e-12)
    D = pd.DataFrame(
        np.abs(Z.values[:,None,:] - Z.values[None,:,:]).mean(2),
        index=F.index, columns=F.index)
    meta = load_meta()
    ROWS = work_rows(meta)           # 000119_001743 → メタデータの行
    LAB  = work_labels()             # 000119_001743 → 中島敦『光と風と夢』
                                     # ラベルは分析対象外の作品も引けるよう全件から

    miss = [s for s in D.index if s not in LAB]
    if miss:
        print(f'[warn] メタデータに無い作品 {len(miss)} 件（ファイル名のまま表示）: '
              + '，'.join(miss[:5]))

    def au(s):
        r = ROWS.get(s)
        return None if r is None else r.get('author_ja')

    pairs = [(a,b,D.loc[a,b]) for i,a in enumerate(D.index) for b in D.index[i+1:]]
    near = pd.DataFrame(
        [{'順位': i+1, 'Delta 距離': d,
          '作品 A': LAB.get(a,a), '作品 B': LAB.get(b,b),
          '同一作家': '●' if au(a) is not None and au(a)==au(b) else ''}
         for i,(a,b,d) in enumerate(sorted(pairs, key=lambda t:t[2])[:12])])
    show(near, caption='最も近い作品ペア（Delta 距離の小さい順・上位12）',
         fmt={'Delta 距離': '{:.4f}'})
    n_same = (near['同一作家'] == '●').sum()
    print(f'上位12ペアのうち同一作家が {n_same} 組。'
          'ここが多いほど，Delta は作家を測っている。')

### 演習 1 — 作家効果 vs 時代効果

最近傍が**同じ作家**である割合と，**同じ時代**である割合を比べよ。
どちらが大きいか。それは何を意味するか。

In [ ]:
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    # キーの 0 埋めと file_v1 の欠損は work_rows() が面倒を見る。
    # ここで自前に辞書を作ると，増補45点の file_v1 が空なので落ちるか，
    # 0 埋めの違いで**1件も一致しないまま割合だけ出る**。
    labs = work_rows(meta)
    have = [f for f in D.index if f in labs]
    if len(have) < len(D.index):
        print(f'[warn] メタデータに無い作品 {len(D.index)-len(have)} 件を除いて集計')
    if not have:
        print('[FATAL] 1件も突合できていない。割合を読んではいけない')
    else:
        # 最近傍は作品ごとに1回求めれば済む（観点ごとに引き直さない）
        nn = {f: D.loc[f, [x for x in have if x != f]].idxmin() for f in have}
        FIELD_JA = {'author_ja':'作家', 'period':'時代', 'genre_sub':'ジャンル',
                    'register_level':'文体の階層', 'narration':'語り'}
        rows = []
        for field, ja in FIELD_JA.items():
            hit = sum(labs[nn[f]][field] == labs[f][field] for f in have)
            rows.append({'観点': ja, '列名': field,
                         '一致': hit, '作品数': len(have),
                         '一致率': hit/len(have)})
        t = show(pd.DataFrame(rows).sort_values('一致率', ascending=False),
                 caption='最近傍が同じ属性を持つ割合（Delta・1近傍）',
                 fmt={'一致率': '{:.1%}'})
        top = t.iloc[0]
        print(f'最も一致しやすいのは「{top["観点"]}」（{top["一致率"]:.1%}）。'
              'これがコーパスで最も強い信号である。')

## 3. 主成分分析 — 何が第1主成分か

MFW の z 行列を主成分分析すると，第1・第2主成分に何が現れるか。
**時代か，ジャンルか，作家か，文語/口語か。**

散布図を4通りの色分けで描き，どの軸が最も分離しているかを見る。

### 色分けは，変数の種類で変える

4面のうち **`period` の面だけ1色相の濃淡**（薄い水色 → 濃紺の5段）で色分けする。
理由は Step 1 の散布図と同じである。

- 時代は**順序のある変数**である。色相環の8色で色分けすると，明治中期が青・
  明治後期が黄のように隣り合う時代が隣り合う色にならず，
  **「時代が下るとどちらへ動くか」という肝心のことが読めない**
- 濃淡なら近い時代が近い色になり，勾配がそのまま目に入る
- 段数は**5段**。1色相の濃淡で順序を見せるには隣の段の明度差が 0.06 以上
  要り，白地の散布図で使える青の幅では6段取るとどこかが見分けられなくなる
  （`year_bands()` の説明を見よ）

色の濃淡だけでは動きの向きが読みにくいので，**各段の中央値を結んだ矢印**を
重ねる。

⚠ **この5段の色は全 Step で同じ時代を指す。** `year_bands()` を通すのは
そのためで，Step 1 の `Step1_bungo_kogo.svg` と並べて読める。

**濃淡を使うのは `period` の面だけである。** `style_class` も順序のある
変数だが，青の濃淡は「時代」を指す約束にしてあるので，別の変数に流用すると
図をまたいだ読みが壊れる。残りの3面は色相で色分けする。

In [ ]:
p = OUT/'descriptive'/'pca_coordinates.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    pc = pd.read_csv(p)

    # **色分けする軸は，図の側で宣言して取りに行く。**
    # 07 の突合が外れると year_first も period も空のまま CSV に入る。
    # それを黙って色分けすると，62 点が「初出年不明」の灰色で並ぶ（実際に起きた）。
    # attach_meta は空のセルもメタデータから補い，補えなかった件数を出す。
    pc = attach_meta(pc, ['year_first','period','style_class',
                          'genre_sub','author_ja'], stem_col='work')

    # 補ってもなお空が残るなら，メタデータ側の空きと突き合わせて言い切る。
    # **凡例の「初出年不明」は，作品の性質ではなく突合の失敗**であることが
    # 多い。両者を混同しないために，ここで数える。
    _yr = pd.to_numeric(pc.year_first, errors='coerce')
    n_nan = int(_yr.isna().sum())
    if n_nan:
        _m = load_meta()
        n_meta_blank = int(pd.to_numeric(_m.year_first, errors='coerce')
                           .isna().sum())
        print(f'[warn] 初出年が読めない作品が {n_nan}/{len(pc)} 件ある。'
              f'メタデータ側で空なのは {n_meta_blank} 件。')
        if n_nan > n_meta_blank:
            print('       **差は突合の失敗である。** メタデータには年がある。')
            print('       07_descriptive_stats.py を今の版で走らせ直すこと'
                  '（キーの 0 埋めの綴り違いを直してある）。')
            print('       それでも残るなら '
                  'results/descriptive/meta_unmatched.csv を見る。')

    fig, axes = plt.subplots(2,2, figsize=(13,10))

    # ---- 第1面：時代は「順序」なので1色相の濃淡で色分けする ---------------------
    ax = axes[0,0]
    code, blabels, bcolors = year_bands(pc.year_first)
    # 初出年が読めないものは -1。**いちばん古い段に混ぜない**（灰で別扱い）
    unk = code < 0
    if unk.any():
        ax.scatter(pc.PC1[unk], pc.PC2[unk], s=22, color='#cccccc',
                   edgecolor='white', linewidth=.5,
                   label=f'初出年不明（{int(unk.sum())}）')
    for b, (lab, col) in enumerate(zip(blabels, bcolors)):
        m = code == b
        if not m.any():
            continue
        ax.scatter(pc.PC1[m], pc.PC2[m], s=42, alpha=.9, color=col,
                   edgecolor='white', linewidth=.5,
                   label=f'{lab}（{int(m.sum())}）')
    # 平均ではなく中央値。外れ値1点で軌跡の向きが変わるのを防ぐ。
    mx = [pc.PC1[code==b].median() for b in range(len(blabels))]
    my = [pc.PC2[code==b].median() for b in range(len(blabels))]
    pts = [(x,y) for x,y in zip(mx,my) if np.isfinite(x) and np.isfinite(y)]
    for (x0,y0),(x1,y1) in zip(pts, pts[1:]):
        ax.annotate('', xy=(x1,y1), xytext=(x0,y0),
                    arrowprops=dict(arrowstyle='-|>', color='#5a5a55', lw=1.2,
                                    shrinkA=7, shrinkB=7, alpha=.9))
    # 面の名は「period」ではなく「初出年の5段」。period 列は6区分だが，
    # 作品数3点の明治前期を明治中期に畳んで5段にしてある（year_bands）。
    ax.set_title('period → 初出年の5段（濃いほど新しい／矢印は段の中央値）')
    # 凡例は点の上に重なりうるので，白の半透明を敷いて文字を読めるようにする
    ax.legend(fontsize=6.5, ncol=1, loc='best', frameon=True,
              framealpha=.85, edgecolor='none')

    # ---- 残り3面：順序の無い変数なので色相で色分けする --------------------------
    for ax, field in zip(axes.ravel()[1:],
                         ['style_class','genre_sub','author_ja']):
        keys = pc[field].value_counts().index[:8]
        for i,k in enumerate(keys):
            d = pc[pc[field]==k]
            ax.scatter(d.PC1, d.PC2, s=38, alpha=.85,
                       color=PALETTE[i%len(PALETTE)], label=str(k)[:18],
                       edgecolor='white', linewidth=.5)
        rest = pc[~pc[field].isin(keys)]
        if len(rest): ax.scatter(rest.PC1, rest.PC2, s=18, color='#cccccc', label='その他')
        ax.set_title(field); ax.legend(frameon=False, fontsize=7, ncol=2)

    for ax in axes.ravel():
        ax.axhline(0,color='#ddd',lw=.8); ax.axvline(0,color='#ddd',lw=.8)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
        ax.spines[['top','right']].set_visible(False)
    fig.suptitle('MFW-PCA：どの軸が第1・第2主成分を説明するか', y=1.01)
    fig.tight_layout(); save_fig(fig, 'Step4_pca_four'); plt.show()

    # 目で見た勾配を数字で裏づける。**図だけで「時代の軸だ」と言わない。**
    def mono(vals):
        v = [x for x in vals if np.isfinite(x)]
        up = all(a <= b for a,b in zip(v, v[1:]))
        dn = all(a >= b for a,b in zip(v, v[1:]))
        return '単調' if (up or dn) else '単調でない'

    yr = pd.to_numeric(pc.year_first, errors='coerce')
    show(pd.DataFrame([
            {'主成分':'PC1', '初出年との ρ': pc.PC1.corr(yr, method='spearman'),
             '段の中央値': mono(mx)},
            {'主成分':'PC2', '初出年との ρ': pc.PC2.corr(yr, method='spearman'),
             '段の中央値': mono(my)}]),
         caption='主成分と初出年の関係（Spearman の順位相関）',
         fmt={'初出年との ρ':'{:+.3f}'})
    print('|ρ| が大きい主成分が「時代の軸」である。段の中央値が'
          '**単調でない**なら，濃淡の勾配は見かけだけかもしれない。')
    print('**いずれにせよ作家効果と交絡している。** 演習 1 の一致率と併せて読む。')

## 4. 特徴語 — 対数尤度比 G²

2つのコーパスで語の頻度を比べる標準的な方法。

$$G^2 = 2\left(a\ln\frac{a}{E_1} + b\ln\frac{b}{E_2}\right)$$

**注意点が3つある。**

1. G² は**標本サイズに比例して大きくなる**。大きいコーパスでは些細な差も有意になる。
   必ず**効果量**（log ratio）を併記する。
2. 1作家に偏った時代では，**その作家の固有名詞**が上位に来る。
   「時代の特徴語」ではなく「特定作家の語彙」を見ていないか確認する。
3. **G² は「どこで多いか」を見ない。** 頻度の合計しか見ないので，
   1作品に固まって出る語と，全作品に薄く広がる語を区別できない。
   これは 4.1 で扱う。

In [ ]:
p = OUT/'descriptive'/'keyness_by_period.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    ky = read_table(p)
    # 散らばりの列は 07 が書く（df_all_prop / dp_in / top_work_share）。
    # 古い CSV には無いので，無ければ 4.1 以降が動かないことを先に言う。
    DISP = ['df_all_prop', 'df_in_prop', 'dp_in', 'top_work_share', 'top_work']
    lack = [c for c in DISP if c not in ky.columns]
    if lack:
        print('[warn] 散らばり（ディスパーション）の列が無い: ' + '，'.join(lack))
        print('       古い 07_descriptive_stats.py で作った CSV である。')
        print('       上の「07 を走らせるセル」を実行し直すこと。'
              '（下の G² の表は列が無くても出る）\n')

    # **時代を列にした表にする。** 時代ごとに行を流すと，同じ語が
    # どの時代にも出ていることに気づけない。列に並べれば横に読める。
    TOPK0 = 15
    cols = {}
    for per in sorted(ky.period.unique()):
        d = ky[(ky.period==per)&(ky.G2>0)].nlargest(TOPK0,'G2')
        cols[per] = [f'{r.term} {r.G2:.0f}' for _,r in d.iterrows()] \
                    + [''] * (TOPK0 - len(d))
    kt = pd.DataFrame(cols, index=[f'{i+1}' for i in range(TOPK0)])
    kt.index.name = '順位'
    # 有意語が 15 に満たない時代があると空行が並ぶので，全列が空の行は落とす
    kt = kt[(kt != '').any(axis=1)]
    show(kt.reset_index(), caption=f'時代ごとの特徴語 上位{TOPK0}（語と G²）')

    # 語だけの表は「どの時代にも出る語」を見落としやすいので，
    # 複数の時代で上位に入った語を名指しする。
    shared = Counter(x.split(' ')[0] for v in cols.values() for x in v if x)
    dup = {w: n for w, n in shared.items() if n > 1}
    if dup:
        print('2つ以上の時代で上位に入った語: '
              + '，'.join(f'{w}({n}時代)' for w, n in
                          sorted(dup.items(), key=lambda x: -x[1])))
        print('**同じ語が複数の時代で「特徴語」になるのは，比較の相手が'
              '「残り全部」だからである。**隣の時代とだけ比べれば消えることがある。')

### 4.1 culling — G² が見ていないもの

上のリストは「その時代に頻度が偏っている語」である。しかし
**偏りには2種類ある。**

次の2語を考える。どちらも明治後期の全体で 160 回，他の時代では 0 回とする。

| 語 | 出かた |
|---|---|
| **A** | 明治後期の 8 作品すべてに 20 回ずつ |
| **B** | 明治後期の **1 作品だけ**に 160 回。残りの 7 作品には 0 回 |

G² は総頻度と総語数だけから計算されるので，**この2語の G² は完全に同じ
値になる**（実際に人工データで確かめると，どちらも G²=221.01，
log ratio=18.92 になる）。しかし意味はまったく違う。

- **A は evenly distributed** — 「明治後期の書き方」と呼べる
- **B は bursty** — 「その1作品の語彙」であって時代の特徴ではない

**G² はこの区別をしない。** だから G² のリストを見るだけでは，
時代の特徴を見ているのか，数点の作品を見ているのか分からない。

#### culling とは

そこで**語を絞る**。文体計量でいう *culling*（Eder らの用語）は，
**一定割合の文書に現れない語を特徴量から外す**操作である。

> **culling at 10%** = コーパス全体の文書頻度（df）が 10% 未満の語を弾く

本コーパスは 101 点なので，**11 点未満に出る語を捨てる**ことになる。

⚠ **culling は G² の値を変えない。** 語を1つ落としても，他の語の
`a, b, c, d` は変わらないからである（`c, d` は総語数で，特徴量の集合とは
無関係）。変わるのは**リストの中身と順位**だけである。
「culling したら G² が動いた」なら，語ではなく**トークンを消してしまって
いる**（総語数が変わっている）。その2つは別の操作である。

#### 何で bursty さを測るか

`07_descriptive_stats.py` は次の列を書き出す。**df は粗い指標**
（出たか出ないかの 0/1 しか見ない）なので，併せて読む。

| 列 | 何を測るか | bursty なら |
|---|---|---|
| `df_all_prop` | 全 101 点のうち何割に出るか | **小さい** |
| `df_in_prop` | その時代の作品のうち何割に出るか | 小さい |
| `dp_in` | **その時代の中での** Gries の DP（偏差の総和の半分） | **1 に近い** |
| `top_work_share` | 総頻度のうち最多の1作品が占める割合 | 1 に近い |
| `top_work` | その1作品はどれか | — |

**割合の列はすべて 0–1 である**（`df_all_prop` = 0.1584 は 15.84 % の意）。
このパイプラインでは，0–1 の割合を `_prop` / `_ratio` / `_share`，
0–100 の百分率だけを `_pct` と綴り分ける。名前を見れば尺度が決まる。
表計算ソフトで開いて「0.15」と「15」を取り違えないための約束である。

⚠ `dp_gries`（コーパス全体の DP）を bursty 判定に使わないこと。
**ある時代だけに出る語は，その時代の全作品に均等に出ていても
全体では「偏った」語**になり，DP が 0.5 前後になる。しかしその偏りは
時代の特徴語として**望ましい**偏りである。全体の DP は
「時代に偏る」と「1作品に偏る」を区別できない。分母をその時代に限った
`dp_in` を使う。（上の A は `dp_in`=0.004，B は 0.871 になる。
`dp_gries` はどちらも 0.50 前後で，区別できない。）

In [ ]:
# ---- culling at df < 10% と，しない場合を並べる ------------------------
CULL = 0.10      # コーパス全体の df がこの割合未満の語を弾く
TOPK = 15

p = OUT/'descriptive'/'keyness_by_period.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    ky = read_table(p)
    if 'df_all_prop' not in ky.columns:
        print('[warn] df_all_prop 列が無い（旧名 df_all_pct も無い）。'
              '07 を走らせるセルを実行し直すこと。')
    else:
        # **語幹（000160_003368）では誰の何だか分からない。**
        # 「この語はこの作品のものだ」と言えて初めて bursty の話が通じる。
        LAB4 = work_labels()          # 語幹 → 作家『作品』
        def _w4(x):
            x = str(x)
            return LAB4.get(x, x)     # 引けないときは語幹のまま出す
        for per in sorted(ky.period.unique()):
            a = ky[(ky.period==per)&(ky.G2>0)]
            b = a[a.df_all_prop >= CULL]
            A = list(a.nlargest(TOPK,'G2').term)
            B = list(b.nlargest(TOPK,'G2').term)
            rank = {t:i+1 for i,t in enumerate(a.nlargest(500,'G2').term)}
            sa, sb = set(A), set(B)

            # **左右に並べた表にする。** 2本のリストを print で上下に
            # 並べると，何番目が入れ替わったのかを目で数えることになる。
            rows = []
            for i in range(min(TOPK, max(len(A), len(B)))):
                t_a, t_b = (A[i] if i < len(A) else ''), (B[i] if i < len(B) else '')
                mark = ''
                if t_a and t_a not in sb:
                    r = a[a.term == t_a].iloc[0]
                    mark = (f'← 落ちた（df {r.df_all_prop:.0%}／'
                            f'dp_in {r.dp_in:.2f}／最多 {_w4(r.top_work)} '
                            f'{r.top_work_share:.0%}）')
                elif t_b and t_b not in sa:
                    mark = f'→ 入った（元 {rank.get(t_b, "500+")} 位）'
                rows.append({'順位': i+1, 'そのまま': t_a,
                             'culling 後': t_b, '入れ替わり': mark})
            show(pd.DataFrame(rows),
                 caption=(f'{per}　有意語 {len(a):,} → culling 後 {len(b):,}'
                          f'（{len(b)/max(1,len(a)):.0%} 残る）'
                          f'／閾値 df < {CULL:.0%} を除外'))

### 4.2 どれだけ変わったか — 数えて比べる

目で見た印象では「だいぶ変わった」とも「あまり変わらない」とも言える。
**数えること。** 指標は3種類あればよい。

| 指標 | 定義 | 読み |
|---|---|---|
| **残存率** | culling 後に残る有意語の割合 | 小さい＝その時代の特徴語は薄く分布する語が少ない |
| **重なり@k** | そのままの上位 k 語のうち，culling 後も上位 k 語に残る割合 | 1 に近い＝culling してもリストはほぼ同じ |
| **入れ替わり** | culling 後の上位 k 語のうち，元は上位 k 語に無かった語数 | 大きい＝頻度順では埋もれていた語が浮上した |

⚠ **順位相関（Spearman ρ）を「そのまま vs culling 後」で計算しても
意味がない。** culling は語を落とすだけで残った語の順序を変えないので，
**ρ は必ず 1 になる**。使うなら「G² の順位」と「`dp_in`」の相関を見て，
*このコーパスでは高頻度の特徴語ほど bursty なのか*を問うほうがよい。

In [ ]:
p = OUT/'descriptive'/'keyness_by_period.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    ky = read_table(p)
    if 'dp_in' not in ky.columns:
        print('[warn] dp_in 列が無い。07 を走らせるセルを実行し直すこと。')
    else:
        rows = []
        for per in sorted(ky.period.unique()):
            a = ky[(ky.period==per)&(ky.G2>0)]
            b = a[a.df_all_prop >= CULL]
            rec = {'period': per, '有意語': len(a), 'culling後': len(b),
                   '残存率': len(b)/max(1,len(a))}
            for k in (15, 50):
                A = set(a.nlargest(k,'G2').term)
                B = set(b.nlargest(k,'G2').term)
                # **分母は k ではなく min(k, 実際の語数)。** 有意語が k に
                # 満たない時代（作品数の少ない時代）で，k で割ると
                # 重なりが不当に低く出て「culling で激変した」と読める。
                rec[f'重なり@{k}'] = len(A & B)/max(1, min(k, len(a)))
                rec[f'入替@{k}'] = len(B - A)
            t50 = a.nlargest(50,'G2')
            keep = t50[t50.df_all_prop >= CULL]
            drop = t50[t50.df_all_prop <  CULL]
            rec['残る語のdp_in中位'] = keep.dp_in.median() if len(keep) else np.nan
            rec['落ちる語のdp_in中位'] = drop.dp_in.median() if len(drop) else np.nan
            # G² の順位と bursty さの関係。正なら「上位ほど一点に固まる」
            top = a.nlargest(100,'G2')
            rec['ρ(G²,dp_in)'] = (top.G2.corr(top.dp_in, method='spearman')
                                  if len(top) > 5 else np.nan)
            rows.append(rec)
        mt = pd.DataFrame(rows).rename(columns={'period':'時代'})
        show(mt, caption=f'culling（df < {CULL:.0%} を除外）の効き方',
             fmt={'残存率':'{:.1%}', '重なり@15':'{:.0%}', '重なり@50':'{:.0%}',
                  '残る語のdp_in中位':'{:.3f}', '落ちる語のdp_in中位':'{:.3f}',
                  'ρ(G²,dp_in)':'{:+.2f}'})
        print('重なり@15 が 1.00 に近い時代は，culling の有無で結論が変わらない。')
        print('小さい時代は，**そのままの上位語が数点の作品に依存している**。')

### 4.3 culling の閾値は bursty さの代理にすぎない

`df < 10%` という規則は**文書頻度**を見ているだけで，bursty さを直接
測ってはいない。だから2種類の取りこぼしが起きる。

- **弾かれないが bursty** … 多くの作品に1回ずつ出るうえで，1作品に
  大量に出る語（df は大きいのに `dp_in` も大きい）
- **弾かれるが均等** … その時代の作品数が少ないために df が 10% に
  届かないだけで，その時代の中では均等に出る語

下の図は横軸に `df_all_prop`（culling の規則），縦軸に `dp_in`
（bursty さ）を取る。**規則が捉え損なう語は左上と右下に現れる。**
その語に名前が付いているので，自分の目で確かめられる。

### 図は2つ出る — 静止版（SVG）と対話版（HTML）

名前を付けられるのは数語だけである。300 点に全部名前を付ければ図は読めない。
かといって名前が無ければ，「左上が bursty」と言われても**どの語なのかを
確かめようがない**。そこで同じ図から2つ書き出す。

| ファイル | 用途 |
|---|---|
| `Step4_keyness_dispersion.svg` | 論文・配布用。従来どおり。拡大も加筆も自由 |
| `Step4_keyness_dispersion.html` | 探索用。**点にカーソルを近づけると語が出る** |

HTML は**その SVG をそのまま埋め込んでいる**（描き直していない）ので，
注記も軸も静止版と同一である。加えて

- 最も近い点を拾うので，小さな点の真上に置かなくてよい
- 語・作品・時代で絞り込める検索欄がある
- 全点の表が下に付く（カーソルが使えなくても同じ情報に届く）。
  **表の行にカーソルを乗せると，図の上のその点が光る**
- 「SVG を保存」から静止版を取り出せる

外部の JavaScript ライブラリは使っていない。**ネットワークが塞がれたマシンでも
ブラウザで開ける**（`open my_work/results/Step4_keyness_dispersion.html`）。

In [ ]:
p = OUT/'descriptive'/'keyness_by_period.csv'
if need(p, 'この分析のスクリプトを走らせるセルを先に実行すること'):
    ky = read_table(p)
    if 'dp_in' not in ky.columns:
        print('[warn] dp_in 列が無い。07 を走らせるセルを実行し直すこと。')
    else:
        sub = pd.concat([ky[(ky.period==per)&(ky.G2>0)].nlargest(50,'G2')
                         for per in sorted(ky.period.unique())])
        keep = (sub.df_all_prop >= CULL).values
        # このセル単独でも走るように，語幹→作家『作品』の表はここでも作る
        LAB4 = work_labels()
        def _w4(x):
            x = str(x)
            return LAB4.get(x, x)

        fig, ax = plt.subplots(figsize=(9.4,6.2))
        ax.axvspan(0, CULL, color='#f4f4f1', zorder=0)   # 弾かれる領域
        # 色だけでなくマーカーの形も変える（色覚の多様性と白黒印刷のため）
        ax.scatter(sub.df_all_prop[keep], sub.dp_in[keep], s=26, alpha=.80,
                   color=PALETTE[0], linewidth=0, label=f'残る（df≧{CULL:.0%}）')
        ax.scatter(sub.df_all_prop[~keep], sub.dp_in[~keep], s=30, alpha=.85,
                   color=PALETTE[1], marker='^', linewidth=0,
                   label=f'弾かれる（df<{CULL:.0%}）')
        ax.axvline(CULL, color='#8a8a83', lw=1, ls='--')
        ax.axhline(0.5, color='#8a8a83', lw=1, ls=':')

        ax.set_xlabel('df_all_prop — 全作品のうち何割に出るか（culling の規則）')
        ax.set_ylabel('dp_in — その時代の中での偏り（1 に近いほど bursty）')
        ax.set_title('culling の閾値と bursty さは一致しない（各時代の上位50語）')
        ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.03, 1.03)
        ax.legend(frameon=False, loc='upper right')
        for sp in ('top','right'): ax.spines[sp].set_visible(False)
        # **注記より先に tight_layout を呼ぶ。** あとで呼ぶと軸が動き，
        # 置いたラベルが点からずれる（check_scatter_labels.py が検出する）。
        fig.tight_layout()

        # **規則が取りこぼした語にだけ**名前を付ける。全点に付けると読めず，
        # 無条件に上位6語を拾うと「取りこぼしが無い」ときにも何か名前が
        # 付いてしまい，規則が外れている証拠のように見える。
        # 閾値で切って，外れていなければ何も付けない。
        k1 = sub[keep]
        miss = k1[k1.dp_in > 0.5].nlargest(6,'dp_in')      # 残るのに bursty
        k0 = sub[~keep]
        over = k0[k0.dp_in < 0.3].nsmallest(6,'dp_in')     # 弾かれるのに均等
        lab = pd.concat([miss, over])
        if len(lab):
            label_points(ax, lab.df_all_prop, lab.dp_in, lab.term, fontsize=8)

        # SVG（静止版）と HTML（対話版）を同じ図から出す。
        # **注記は SVG の中にあるので対話版にもそのまま残る。**
        # 残りの点は，指せば語が出る。どの点が何の語かを言えないまま
        # 「左上が bursty」と説明しても，受講生には確かめようがない。
        tips = [{'term': r.term,
                 'fields': [('時代', r.period),
                            ('G²', f'{r.G2:.0f}'),
                            ('df（全作品）', f'{r.df_all_prop:.0%}'),
                            ('df（その時代）', f'{r.df_in_prop:.0%}'),
                            ('dp_in', f'{r.dp_in:.3f}'),
                            ('最多作品の占有', f'{r.top_work_share:.0%}'),
                            ('最多作品', _w4(r.top_work)),
                            ('culling', '残る' if r.df_all_prop >= CULL
                                        else '弾かれる')]}
                for _, r in sub.iterrows()]
        save_interactive(
            fig, ax, 'Step4_keyness_dispersion',
            sub.df_all_prop, sub.dp_in, tips, source=p,
            title='culling の閾値と bursty さは一致しない',
            hint=('点にカーソルを近づけると語が出る（最も近い点を拾うので，'
                  '真上に置かなくてよい）。図の中の注記は静止版と同じもので，'
                  '取りこぼした語だけに付いている。'),
            note=(f'各時代の G² 上位50語／縦線は df {CULL:.0%} の閾値，'
                  f'横の点線は dp_in 0.5。左上＝弾かれないが bursty，'
                  f'右下＝弾かれるが均等。'),
            table_cols=['時代', 'G²', 'df（全作品）', 'dp_in',
                        '最多作品の占有', '最多作品', 'culling'])
        plt.show()

        # 規則が外れた語を表にする（どの作品のせいかまで載せる）
        bad = pd.concat([
            miss.assign(位置='左上：弾かれないが bursty'),
            over.assign(位置='右下：弾かれるが均等')])
        if len(bad):
            bad = bad.assign(top_work=bad.top_work.map(_w4))
            show(bad[['位置','period','term','df_all_prop','dp_in',
                      'top_work_share','top_work','G2']]
                 .rename(columns={'period':'時代','term':'語',
                                  'df_all_prop':'df','dp_in':'dp_in',
                                  'top_work_share':'最多作品の占有',
                                  'top_work':'最多作品','G2':'G²'}),
                 caption='df の規則が取りこぼした語',
                 fmt={'df':'{:.0%}', 'dp_in':'{:.2f}',
                      '最多作品の占有':'{:.0%}', 'G²':'{:.0f}'})
            print('**この語については規則が外れている。** 閾値を変えるか，'
                  'dp_in で直接絞ることを検討する。')
        else:
            print(f'取りこぼしなし（df≧{CULL:.0%} の語はどれも dp_in≦0.5，'
                  '弾かれる語はどれも dp_in≧0.3）。')
            print('このコーパスでは df の規則が bursty さの代理として働いている。')

### 演習 2 — bursty か evenly-distributed か

`CULL` を `0.05` / `0.10` / `0.25` と変え，時代ごとに次を答えよ。

1. **重なり@15 が最も小さい時代**はどれか。その時代の
   「落ちた語」の `top_work` を見て，**どの作品が効いていたか**を言え。
2. 「入った語」（culling で浮上した語）を5語選び，それが
   **時代の特徴として説明できるか**を述べよ。説明できないなら，
   何を拾ってしまっているのか。
3. 自分の研究上の問いに対して，**culling すべきか，すべきでないか**。
   *その時代に何が書かれたか*を問うなら bursty な語も証拠になるが，
   *その時代の書き方*を問うなら邪魔になる。**問いによって答えが変わる**
   ことを，具体的な語を挙げて論じよ。
4. ⚠ 閾値は**測る前に決めておく**こと。3通り試してから「いちばん
   きれいな」閾値を選ぶのは，結果に合わせて規則を選ぶことである。
   試すのは**感度分析**として行い，本分析の閾値は先に宣言して報告に書く。

### 演習 3 — 固有名詞を除くとどう変わるか

`data/tokens/tsv/` には品詞情報がある。固有名詞を除いて特徴語を再計算し，
上のリストと比べよ。**消えた語と残った語の違いは何か。**

In [ ]:
# 固有名詞を除いたトークン列を作る（演習用）
def tokens_without_propn(tsv_path):
    out = []
    # 05 は BOM 付きで書く（Excel 対策）ので utf-8-sig で読む
    with open(tsv_path, encoding='utf-8-sig') as fh:
        next(fh); next(fh)           # コメント行とヘッダ
        for line in fh:
            f = line.rstrip('\n').split('\t')
            if len(f) < 13 or f[6] in ('EOS','補助記号','空白'):
                continue
            if f[6]=='名詞' and f[7]=='固有名詞':
                continue
            out.append(f[2])
    return out

tsvs = sorted((TOK/'tsv').glob('*.tsv'))[:3]
for t in tsvs:
    toks = tokens_without_propn(t)
    print(f'{t.name:<24} 固有名詞を除いた語数 {len(toks):,}')
print('\n→ 全件で作り直して 07_descriptive_stats.py にかけ直すのが課題。')

## 5. このステップの課題

次の設問への答えを，テンプレート `my_work/results/Step4_report.md` に書いて提出する（**全体で600–1000字程度**。図表と「再現のための情報」は字数に含めない）。

- **提出先**：Zulip（dh-uosaka.zulipchat.com）の非公開チャネル **2026年度テクスト分析論B** ＞ トピック **Step 4**
- テンプレートの中身をメッセージに貼り付け，図（SVG）・表（CSV）は**同じメッセージに添付**する（1人1通）
- 図は番号で言及し（図1），**図を見なくても論旨が追えるように**書く（SVG は Zulip で表示されないことがある）
- 再提出は元の投稿を直さず，同じトピックに新しく投稿する（手順書 §5.3）

1. TTR・Guiraud R・Yule K・エントロピーを作品長に対してプロットし，
   **通時比較に使うならどれか**を根拠とともに選ぶこと。
2. Delta の最近傍一致率（作家 / 時代 / ジャンル）を報告し，
   このコーパスで通時的主張をするときの注意点を述べること。
3. 固有名詞を除いた特徴語リストを作り，除く前と比較すること。
4. MFW の語数（100 / 300 / 1000）を変えると PCA の布置がどう変わるか確かめること。
5. `speech_markup` が `full` でない作品を除いた場合と含めた場合とで，
   会話文比率と第1主成分の相関がどう変わるかを示すこと。**該当作品は
   `conversion_report.csv` から自分で数えること**（コーパスの増補で
   件数は変わる）。
6. **culling（4.1–4.3）を報告の形に書くこと。** 閾値を1つ宣言し，
   時代ごとに「残存率・重なり@15・入替@15」の表を出し，
   *culling の有無で結論が変わる時代*があるかを述べること。
   変わる時代があれば，**その原因になっている作品を名指しする**
   （`top_work` 列）。加えて，自分の問いに対してどちらを本分析に
   採るかを1段落で正当化すること。
### このステップの到達点（次へ進む条件）

- MFW 行列・Delta 行列・PCA 図が `results/descriptive/` に出ている
- Delta の最近傍が作家・時代・ジャンルのどれと一致しやすいかを言える
- 特徴語リストの上位語について，**それが何を測っているか**を説明できる
- **G² の順位と bursty さが別物であることを，自分のデータの語で示せる**
  （`Step4_keyness_dispersion.svg` の左上・右下の語）
